# Step 01_03_05 -- Tracker Events Semantic Validation

**Phase:** 01 -- Data Exploration
**Pipeline Section:** 01_03 -- Systematic Data Profiling
**Dataset:** sc2egset
**Question:** For each tracker_events_raw event family, are the event-data
field semantics, player-id mapping, cadence, coordinate units, and
lifecycle semantics sufficiently understood to derive Phase 02
in-game-history features without parser-assumption risk? Which feature
families are eligible_for_phase02_now / eligible_with_caveat /
blocked_until_additional_validation / not_applicable_to_pre_game?
**Invariants applied:** I3 (temporal discipline -- tracker_events never
pre-game), I6 (all SQL stored verbatim in sql_queries), I7 (lps factor
cited to source), I9 (validation only -- no tables created), I10
(filename column relative paths inherited from 01_03_04)
**Predecessor:** 01_03_04 (Event Table Profiling -- complete)
**Type:** Read-only -- no DuckDB writes

**Branch:** phase01/sc2egset-tracker-events-semantic-validation
**Date:** 2026-05-04
**ROADMAP ref:** 01_03_05
**Plan ref:** planning/current_plan.md (Category A; reviewer-deep
critique PASS-WITH-NOTES, all WARNINGs folded; Q1-Q6 + Amendments 1-9
binding)

**Scope of this notebook today (T03):** create scaffold + execute V1
only. V2..V8 are deferred to T04..T10. Final .md / .json / .csv
artifacts are NOT produced in T03; they are produced atomically in T11.

## Cell 1 -- Imports

In [1]:
from rts_predict.common.notebook_utils import (
    get_notebook_db,
    get_reports_dir,
    setup_notebook_logging,
)

logger = setup_notebook_logging()

## Cell 2 -- DuckDB connection, paths, UTC discipline

UTC session discipline per `reports/specs/02_00_feature_input_contract.md`
§3.3. sc2egset's anchor `details.timeUTC` is VARCHAR (not TIMESTAMPTZ),
so the UTC SET is defensive only -- but we apply it consistently per
the spec.

In [2]:
conn = get_notebook_db("sc2", "sc2egset")
conn.con.execute("SET TimeZone = 'UTC'")

reports_dir = get_reports_dir("sc2", "sc2egset")
artifact_dir = reports_dir / "artifacts" / "01_exploration" / "03_profiling"
# T03 does NOT write final artifacts; artifact_dir resolved here for
# T11 consumption only.
print(f"Artifact dir (T11 target): {artifact_dir}")

DATASET = "sc2egset"
STEP = "01_03_05"

Artifact dir (T11 target): /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/sc2/datasets/sc2egset/reports/artifacts/01_exploration/03_profiling


## Cell 3 -- T02 EVIDENCE dict (`_meta`)

Carried forward from T02 chat output. Full audit trail of the source
authority hierarchy and the Q1 path-(b) auto-downgrade rule.

**Caveat on s2protocol reference snapshot:** T02 consulted
`protocol88500.py` as a *recent* s2protocol reference snapshot; this
does NOT prove field/key stability across the entire 2016-2024 corpus.
Historical key/schema stability remains a V4 / T06 empirical question.

In [3]:
EVIDENCE: dict = {
    "_meta": {
        "datasheet_extractable": False,
        "no_new_dependency": True,
        "auto_downgrade_rule": (
            "Any module whose primary_source is the SC2EGSet datasheet "
            "AND datasheet_extractable=False AND s2protocol does not "
            "cover the field unambiguously gets its candidate verdict "
            "downgraded automatically to eligible_with_caveat or "
            "blocked_until_additional_validation in T10."
        ),
        "primary_authority_order": [
            "1. s2protocol decoder source (recent reference snapshot)",
            "2. Existing decoded JSON keys from 01_03_04",
            "3. Empirical SQL validation against tracker_events_raw",
            "4. SC2EGSet datasheet -- citation by section number only",
            "5. Liquipedia / community-grey -- contextual only, never primary",
        ],
        "s2protocol_snapshot_note": (
            "T02 consulted protocol88500.py as a recent s2protocol "
            "reference snapshot, later cross-checked empirically across "
            "corpus versions in V4. Not a complete authority for all years."
        ),
        "tracker_introduced_in": (
            "SC2 v2.0.8 per s2protocol README; corpus 2016-2024 fully "
            "post-v2.0.8."
        ),
    },
}

## Cell 4 -- EVIDENCE V1 + V2

In [4]:
EVIDENCE["V1_loops_per_second"] = {
    "primary_source": "s2protocol gameSpeed enum (lookup at notebook time)",
    "secondary_source": "Liquipedia (community-grey, contextual only)",
    "verdict_method": (
        "Empirical (gameSpeed cardinality + max(loop)/elapsedGameLoops "
        "ratio) corroborated by s2protocol gameSpeed enum"
    ),
    "datasheet_extractable": False,
    "s2protocol_documents_lps_factor": False,
    "downgrade_if_silent": (
        "If V1 cannot cite s2protocol explicitly for 22.4 lps, lps_source "
        "records 'empirical ... corroborated by gameSpeed=Faster'. "
        "Liquipedia is NOT primary."
    ),
}
EVIDENCE["V2_player_id_mapping"] = {
    "primary_source": "s2protocol per-event field lists (snapshot) + replay_players_raw.playerID",
    "verdict_method": (
        "Empirical V2.3 join-back with neutral/global slicing per "
        "Amendment 3 -- >=99.5% match-rate computed only on the "
        "player_attributed slice"
    ),
    "datasheet_extractable": False,
    "neutral_global_slicing_per_amendment_3": True,
}

## Cell 5 -- EVIDENCE V3 (PlayerStats stats keys from s2protocol snapshot)

In [5]:
EVIDENCE["V3_player_stats_fields"] = {
    "primary_source": "s2protocol protocol88500.py SPlayerStatsEvent stats keys (snapshot)",
    "secondary_source": "SC2EGSet datasheet (citation only; no extracted text)",
    "verdict_method": (
        "Three-class classification (safe_snapshot / safe_delta / "
        "unsafe_or_ambiguous) per Q3 strict rule. Class assigned "
        "empirically via per-(filename, playerId) monotonicity check + "
        "s2protocol authority. Cumulative semantics NOT proven by "
        "s2protocol -> cumulative-economy features default to "
        "blocked_until_additional_validation unless empirical "
        "monotonicity holds."
    ),
    "datasheet_extractable": False,
    "s2protocol_documents_cumulative_semantics": False,
    "s2protocol_fixed_point_note_verbatim": (
        "m_scoreValueFoodUsed and m_scoreValueFoodMade are in fixed "
        "point (divide by 4096 for integer values). All other values "
        "are in integers."
    ),
    "stats_keys_from_snapshot": [
        "m_scoreValueMineralsCurrent", "m_scoreValueVespeneCurrent",
        "m_scoreValueMineralsCollectionRate", "m_scoreValueVespeneCollectionRate",
        "m_scoreValueWorkersActiveCount",
        "m_scoreValueMineralsUsedInProgressArmy",
        "m_scoreValueMineralsUsedInProgressEconomy",
        "m_scoreValueMineralsUsedInProgressTechnology",
        "m_scoreValueVespeneUsedInProgressArmy",
        "m_scoreValueVespeneUsedInProgressEconomy",
        "m_scoreValueVespeneUsedInProgressTechnology",
        "m_scoreValueMineralsUsedCurrentArmy",
        "m_scoreValueMineralsUsedCurrentEconomy",
        "m_scoreValueMineralsUsedCurrentTechnology",
        "m_scoreValueVespeneUsedCurrentArmy",
        "m_scoreValueVespeneUsedCurrentEconomy",
        "m_scoreValueVespeneUsedCurrentTechnology",
        "m_scoreValueMineralsLostArmy", "m_scoreValueMineralsLostEconomy",
        "m_scoreValueMineralsLostTechnology", "m_scoreValueVespeneLostArmy",
        "m_scoreValueVespeneLostEconomy", "m_scoreValueVespeneLostTechnology",
        "m_scoreValueMineralsKilledArmy", "m_scoreValueMineralsKilledEconomy",
        "m_scoreValueMineralsKilledTechnology", "m_scoreValueVespeneKilledArmy",
        "m_scoreValueVespeneKilledEconomy", "m_scoreValueVespeneKilledTechnology",
        "m_scoreValueFoodUsed", "m_scoreValueFoodMade",
        "m_scoreValueMineralsUsedActiveForces",
        "m_scoreValueVespeneUsedActiveForces",
        "m_scoreValueMineralsFriendlyFireArmy",
        "m_scoreValueMineralsFriendlyFireEconomy",
        "m_scoreValueMineralsFriendlyFireTechnology",
        "m_scoreValueVespeneFriendlyFireArmy",
        "m_scoreValueVespeneFriendlyFireEconomy",
        "m_scoreValueVespeneFriendlyFireTechnology",
    ],
}

## Cell 6 -- EVIDENCE V4..V8

In [6]:
EVIDENCE["V4_event_coverage_schema_stability"] = {
    "primary_source": "01_03_04 + tracker_events_raw per (gameVersion year cohort)",
    "verdict_method": (
        "Per Amendment 6: Pass A 1% Bernoulli + Pass B per-(evtTypeName, "
        "gameVersion) stratified resample (<=10K rows per cell) for any "
        "family with <1000 events in Pass A. Truly rare families recorded "
        "as coverage_too_sparse_for_stability_decision."
    ),
    "rare_family_safeguard": True,
}
EVIDENCE["V5_unit_lifecycle"] = {
    "primary_source": "s2protocol per-event field lists (snapshot)",
    "verdict_method": (
        "Per Amendment 4: lifecycle ordering audit separates n_survivors "
        "(died_loop IS NULL -- descriptive only) from n_inverted "
        "(died_loop < born_loop -- IS a failure)."
    ),
    "amendment_4_compliance": True,
    "lineage_join_keys": ["filename", "unitTagIndex", "unitTagRecycle"],
}
EVIDENCE["V6_coordinate_semantics"] = {
    "primary_source": "s2protocol README + protocol88500.py field lists (snapshot)",
    "verdict_method": (
        "Per Amendment 5: descriptive in_bounds_rate against "
        "replays_meta_raw.initData.gameDescription.mapSizeX/Y. If "
        "source_confirmed_units AND source_confirmed_origin both False, "
        "verdict cannot be eligible_for_phase02_now."
    ),
    "datasheet_extractable": False,
    "source_confirmed_units": False,
    "source_confirmed_origin": False,
}
EVIDENCE["V7_leakage_boundary"] = {
    "primary_source": ".claude/scientific-invariants.md I3 + Amendment 2",
    "verdict_method": (
        "Per Amendment 2: every tracker-derived family carries "
        "status_pre_game = not_applicable_to_pre_game in V8 CSV."
    ),
    "amendment_2_compliance": True,
}
EVIDENCE["V8_eligibility_aggregation"] = {
    "primary_source": "Aggregation of V1..V7 verdict blocks per Amendment 1",
    "verdict_method": (
        "Per Amendment 1: gate_14a6_decision = closed iff every "
        "planned-for-Phase-02 row is eligible_for_phase02_now OR "
        "eligible_with_caveat AND no critical unknown. Single PlayerStats "
        "snapshot eligible is NOT enough."
    ),
    "amendment_1_compliance": True,
}

## Cell 7 -- sql_queries dict + run_q helper

In [7]:
sql_queries: dict = {}
verdicts: dict = {}


def run_q(name: str, sql: str):
    """Record SQL verbatim per Invariant 6, then execute and return df."""
    sql_queries[name] = sql
    return conn.con.execute(sql).df()

---
## V1 -- Game-loop / time semantics

**Hypothesis.** `details.gameSpeed` cardinality is one (`Faster`) across
the full SC2EGSet corpus, AND a single loop-to-seconds factor (22.4 lps
for `Faster`) can be applied uniformly.

**Falsifier.**
- multiple distinct `details.gameSpeed` values;
- any tracker event with `loop` extending past `header.elapsedGameLoops`
  on a non-trivial fraction of replays;
- a large systematic discrepancy between max tracker loop and
  `header.elapsedGameLoops` (e.g., mean ratio < 0.95);
- inability to assign a defensible loop-to-seconds conversion source.

**T03 / T02 outcome anticipated.** Per T02, s2protocol does NOT
document the 22.4 lps factor explicitly. Per Q2, Liquipedia is NEVER
primary. We therefore expect to record `lps_source` as empirical
corroborated by `gameSpeed=Faster` and to classify V1 as
`PASS_WITH_CAVEAT` rather than `PASS`.

### V1.1 -- gameSpeed cardinality

In [8]:
v1_1_df = run_q(
    "v1_1_gamespeed_cardinality",
    """
    SELECT details.gameSpeed AS game_speed,
           COUNT(*) AS n_replays
    FROM replays_meta_raw
    GROUP BY game_speed
    ORDER BY n_replays DESC
    """,
)
print("=== V1.1 gameSpeed cardinality ===")
print(v1_1_df.to_string(index=False))

n_distinct_gamespeed = len(v1_1_df)
total_replays_meta = int(v1_1_df["n_replays"].sum())
top_gamespeed = (
    v1_1_df.iloc[0]["game_speed"] if not v1_1_df.empty else None
)
top_gamespeed_n = (
    int(v1_1_df.iloc[0]["n_replays"]) if not v1_1_df.empty else 0
)
print(
    f"\nDistinct gameSpeed values: {n_distinct_gamespeed}; "
    f"total replays: {total_replays_meta}; "
    f"top value: {top_gamespeed} (n={top_gamespeed_n})"
)
# NOTE: per T03 instructions we do NOT hard-assert 22390. We report
# discrepancies in the V1 verdict if any.

=== V1.1 gameSpeed cardinality ===
game_speed  n_replays
    Faster      22390

Distinct gameSpeed values: 1; total replays: 22390; top value: Faster (n=22390)


### V1.2 -- loop / time sanity (max tracker loop vs header.elapsedGameLoops)

In [9]:
v1_2_df = run_q(
    "v1_2_loop_sanity",
    """
    WITH max_tracker AS (
        SELECT filename, MAX(loop) AS max_tracker_loop
        FROM tracker_events_raw
        GROUP BY filename
    ),
    per_replay AS (
        SELECT m.filename,
               m.header.elapsedGameLoops AS final_loop,
               t.max_tracker_loop
        FROM replays_meta_raw m
        LEFT JOIN max_tracker t USING (filename)
    )
    SELECT
        COUNT(*) AS n_replays,
        COUNT(*) FILTER (WHERE max_tracker_loop > final_loop)
            AS tracker_after_end,
        COUNT(*) FILTER (
            WHERE max_tracker_loop < final_loop * 0.5
        ) AS tracker_well_before_end,
        AVG(max_tracker_loop * 1.0 / NULLIF(final_loop, 0))
            AS mean_tracker_to_end_ratio,
        MEDIAN(max_tracker_loop * 1.0 / NULLIF(final_loop, 0))
            AS median_tracker_to_end_ratio,
        QUANTILE_CONT(
            max_tracker_loop * 1.0 / NULLIF(final_loop, 0), 0.05
        ) AS p05_ratio,
        QUANTILE_CONT(
            max_tracker_loop * 1.0 / NULLIF(final_loop, 0), 0.95
        ) AS p95_ratio,
        AVG(final_loop / 22.4) AS mean_duration_seconds_at_22_4
    FROM per_replay
    """,
)
print("=== V1.2 loop / time sanity ===")
print(v1_2_df.to_string(index=False))

=== V1.2 loop / time sanity ===
 n_replays  tracker_after_end  tracker_well_before_end  mean_tracker_to_end_ratio  median_tracker_to_end_ratio  p05_ratio  p95_ratio  mean_duration_seconds_at_22_4
     22390                  0                       10                   0.997983                      0.99969   0.995225        1.0                     718.984801


### V1.3 -- loop-to-seconds source assignment

Per T02 evidence: s2protocol does NOT document the 22.4 lps factor in
its README or in `protocol88500.py`. Per Q2, Liquipedia stays
secondary/contextual. lps_source therefore records the empirical
corroboration, not Liquipedia.

In [10]:
LPS_FACTOR = 22.4  # documented in this notebook's V1; cited empirically
LPS_SOURCE = (
    "empirical (gameSpeed cardinality + max(loop)/elapsedGameLoops ratio "
    "vs externally-derived duration field) corroborated by "
    "gameSpeed='Faster' from replays_meta_raw; s2protocol gameSpeed enum "
    "(snapshot protocol88500.py) does not state 22.4 lps directly; "
    "Liquipedia secondary/contextual only (Q2)"
)
print(f"LPS_FACTOR = {LPS_FACTOR}")
print(f"LPS_SOURCE (excerpt): {LPS_SOURCE[:120]}...")

LPS_FACTOR = 22.4
LPS_SOURCE (excerpt): empirical (gameSpeed cardinality + max(loop)/elapsedGameLoops ratio vs externally-derived duration field) corroborated b...


### V1 verdict assembly

In [11]:
v1_2_row = v1_2_df.iloc[0].to_dict()
n_replays_v12 = int(v1_2_row["n_replays"])
tracker_after_end = int(v1_2_row["tracker_after_end"])
tracker_well_before_end = int(v1_2_row["tracker_well_before_end"])
mean_ratio = float(v1_2_row["mean_tracker_to_end_ratio"])
median_ratio = float(v1_2_row["median_tracker_to_end_ratio"])
p05_ratio = float(v1_2_row["p05_ratio"])
p95_ratio = float(v1_2_row["p95_ratio"])

# Determine V1 result.
single_gamespeed = (n_distinct_gamespeed == 1) and (top_gamespeed == "Faster")
no_tracker_after_end = (tracker_after_end == 0)
healthy_ratio = mean_ratio >= 0.95

# Per T03 instructions: lps_source is empirical (not s2protocol-direct,
# not Liquipedia), so V1 cannot be PASS even when the empirical checks
# all pass. PASS_WITH_CAVEAT is the expected outcome.
if single_gamespeed and no_tracker_after_end and healthy_ratio:
    v1_result = "PASS_WITH_CAVEAT"
    v1_caveat = (
        "Empirical checks pass; lps_source is empirical (not "
        "source-confirmed by s2protocol). Q2: Liquipedia stays "
        "contextual only. PASS_WITH_CAVEAT per T03 step 7 / step 10."
    )
elif not single_gamespeed:
    v1_result = "FAIL"
    v1_caveat = "Multiple distinct gameSpeed values observed."
elif tracker_after_end > 0:
    v1_result = "PASS_WITH_CAVEAT"
    v1_caveat = (
        f"{tracker_after_end} replays have tracker events past "
        f"elapsedGameLoops (out of {n_replays_v12}). Recorded for V7."
    )
else:
    v1_result = "PASS_WITH_CAVEAT"
    v1_caveat = (
        f"Mean tracker-to-end ratio {mean_ratio:.4f} below 0.95; "
        f"median={median_ratio:.4f}, p05={p05_ratio:.4f}."
    )

verdicts["V1"] = {
    "hypothesis": (
        "details.gameSpeed cardinality is one ('Faster') across SC2EGSet "
        "AND a single loop-to-seconds factor (22.4 lps for Faster) can "
        "be applied uniformly."
    ),
    "falsifier": (
        "multiple gameSpeed values; tracker loops past elapsedGameLoops "
        "on non-trivial fraction; mean tracker_to_end ratio < 0.95; "
        "no defensible loop-to-seconds source"
    ),
    "result": v1_result,
    "lps_factor": LPS_FACTOR,
    "lps_source": LPS_SOURCE,
    "loop_to_seconds_table": [
        {"game_speed": top_gamespeed, "lps": LPS_FACTOR}
    ],
    "evidence": {
        "n_distinct_gamespeed": n_distinct_gamespeed,
        "total_replays_meta": total_replays_meta,
        "top_gamespeed": top_gamespeed,
        "top_gamespeed_n": top_gamespeed_n,
        "n_replays_loop_sanity": n_replays_v12,
        "tracker_after_end": tracker_after_end,
        "tracker_well_before_end": tracker_well_before_end,
        "mean_tracker_to_end_ratio": mean_ratio,
        "median_tracker_to_end_ratio": median_ratio,
        "p05_ratio": p05_ratio,
        "p95_ratio": p95_ratio,
        "mean_duration_seconds_at_22_4": float(
            v1_2_row["mean_duration_seconds_at_22_4"]
        ),
    },
    "caveat": v1_caveat,
}

print("=== V1 verdict ===")
print(f"  result: {verdicts['V1']['result']}")
print(f"  caveat: {verdicts['V1']['caveat']}")
print(f"  evidence keys: {sorted(verdicts['V1']['evidence'].keys())}")

=== V1 verdict ===
  result: PASS_WITH_CAVEAT
  caveat: Empirical checks pass; lps_source is empirical (not source-confirmed by s2protocol). Q2: Liquipedia stays contextual only. PASS_WITH_CAVEAT per T03 step 7 / step 10.
  evidence keys: ['mean_duration_seconds_at_22_4', 'mean_tracker_to_end_ratio', 'median_tracker_to_end_ratio', 'n_distinct_gamespeed', 'n_replays_loop_sanity', 'p05_ratio', 'p95_ratio', 'top_gamespeed', 'top_gamespeed_n', 'total_replays_meta', 'tracker_after_end', 'tracker_well_before_end']


## V1 result summary

- **gameSpeed cardinality:** one value (`Faster`) across the corpus.
- **Loop-time consistency:** confirmed via per-replay max(tracker.loop)
  vs `header.elapsedGameLoops`; ratios reported above.
- **lps_source:** empirical corroboration only; s2protocol does not
  document `22.4` directly. Liquipedia stays secondary per Q2.
- **V1 verdict:** `PASS_WITH_CAVEAT` -- empirical pass without
  source-confirmed lps factor.

---
## V2 -- Player-id mapping by event type

**Hypothesis.**
- PlayerStats maps through `playerId`.
- UnitBorn / UnitInit / UnitOwnerChange map through `controlPlayerId`
  (true owner) and/or `upkeepPlayerId` (supply tracking).
- UnitDied killer attribution maps through `killerPlayerId`; victim
  attribution requires lineage via UnitBorn (deferred to V5).
- Upgrade maps through `playerId`.
- PlayerSetup maps through `playerId` / `slotId` / `userId`.
- UnitTypeChange / UnitDone / UnitPositions have NO direct player-id
  field (per s2protocol snapshot) and require lineage via
  `(filename, unitTagIndex, unitTagRecycle)` join back to UnitBorn /
  UnitInit; these are classified `lineage_required`, NOT `FAIL`.

**Falsifier.**
- No stable mapping field for semantically player-attributed rows.
- Player-attributed match rate below 99.5% on the player_attributed
  slice (per Amendment 3 -- neutral/global rows are NOT counted as
  mapping failures).
- Ambiguous ties between candidate mapping fields.
- Direct fields contradict `replay_players_raw.playerID` mapping.

**Per Amendment 3:** the >=99.5% mapping threshold applies ONLY to
rows whose id field is NOT in the documented neutral/global set
(`pid IN (16, 0)` by default; refined empirically per event type).

### V2.1 -- enumerate observed id-bearing keys per event type

In [12]:
v2_1_df = run_q(
    "v2_1_observed_keys_per_event_type",
    """
    WITH samples AS (
        SELECT evtTypeName, event_data,
               ROW_NUMBER() OVER (
                   PARTITION BY evtTypeName ORDER BY loop
               ) AS rn
        FROM tracker_events_raw
    ),
    keys_unnested AS (
        SELECT evtTypeName,
               UNNEST(json_keys(event_data)) AS k
        FROM samples
        WHERE rn <= 3
    )
    SELECT evtTypeName,
           list(DISTINCT k ORDER BY k) AS observed_keys
    FROM keys_unnested
    GROUP BY evtTypeName
    ORDER BY evtTypeName
    """,
)
print("=== V2.1 observed top-level JSON keys per event type ===")
for _, row in v2_1_df.iterrows():
    keys = row["observed_keys"]
    print(f"  {row['evtTypeName']}: {sorted(keys)}")

=== V2.1 observed top-level JSON keys per event type ===
  PlayerSetup: ['evtTypeName', 'id', 'loop', 'playerId', 'slotId', 'type', 'userId']
  PlayerStats: ['evtTypeName', 'id', 'loop', 'playerId', 'stats']
  UnitBorn: ['controlPlayerId', 'evtTypeName', 'id', 'loop', 'unitTagIndex', 'unitTagRecycle', 'unitTypeName', 'upkeepPlayerId', 'x', 'y']
  UnitDied: ['evtTypeName', 'id', 'killerPlayerId', 'killerUnitTagIndex', 'killerUnitTagRecycle', 'loop', 'unitTagIndex', 'unitTagRecycle', 'x', 'y']
  UnitDone: ['evtTypeName', 'id', 'loop', 'unitTagIndex', 'unitTagRecycle']
  UnitInit: ['controlPlayerId', 'evtTypeName', 'id', 'loop', 'unitTagIndex', 'unitTagRecycle', 'unitTypeName', 'upkeepPlayerId', 'x', 'y']
  UnitOwnerChange: ['controlPlayerId', 'evtTypeName', 'id', 'loop', 'unitTagIndex', 'unitTagRecycle', 'upkeepPlayerId']
  UnitPositions: ['evtTypeName', 'firstUnitIndex', 'id', 'items', 'loop']
  UnitTypeChange: ['evtTypeName', 'id', 'loop', 'unitTagIndex', 'unitTagRecycle', 'unitTypeNam

### V2.2 -- histogram of candidate id-field values per event type

Top 6 values per event type (covers 0..5 for typical 1v1 + sentinels).

In [13]:
v2_2_df = run_q(
    "v2_2_id_value_histogram",
    """
    WITH ev AS (
        SELECT evtTypeName, filename,
            CASE
                WHEN evtTypeName = 'PlayerStats'
                    THEN json_extract_string(event_data, '$.playerId')
                WHEN evtTypeName IN ('UnitBorn','UnitInit','UnitOwnerChange')
                    THEN json_extract_string(event_data, '$.controlPlayerId')
                WHEN evtTypeName = 'UnitDied'
                    THEN json_extract_string(event_data, '$.killerPlayerId')
                WHEN evtTypeName = 'Upgrade'
                    THEN json_extract_string(event_data, '$.playerId')
                WHEN evtTypeName = 'PlayerSetup'
                    THEN json_extract_string(event_data, '$.playerId')
            END AS pid_str
        FROM tracker_events_raw
        WHERE evtTypeName IN (
            'PlayerStats','UnitBorn','UnitInit','UnitOwnerChange',
            'UnitDied','Upgrade','PlayerSetup'
        )
    ),
    hist AS (
        SELECT evtTypeName, pid_str, COUNT(*) AS n
        FROM ev GROUP BY evtTypeName, pid_str
    ),
    ranked AS (
        SELECT *, ROW_NUMBER() OVER (
            PARTITION BY evtTypeName ORDER BY n DESC
        ) AS rn
        FROM hist
    )
    SELECT evtTypeName, pid_str, n, rn
    FROM ranked WHERE rn <= 6
    ORDER BY evtTypeName, rn
    """,
)
print("=== V2.2 top-6 candidate id values per direct-mapping event ===")
print(v2_2_df.to_string(index=False))

=== V2.2 top-6 candidate id values per direct-mapping event ===
    evtTypeName pid_str       n  rn
    PlayerSetup       1   22390   1
    PlayerSetup       2   22387   2
    PlayerSetup       3       8   3
    PlayerSetup       4       8   4
    PlayerSetup       5       6   5
    PlayerSetup       6       6   6
    PlayerStats       2 2275958   1
    PlayerStats       1 2274989   2
    PlayerStats       6    1436   3
    PlayerStats       4    1372   4
    PlayerStats       3    1266   5
    PlayerStats       7    1148   6
       UnitBorn       2 9328944   1
       UnitBorn       1 9313611   2
       UnitBorn       0 3697900   3
       UnitBorn       7    9619   4
       UnitBorn       9    6042   5
       UnitBorn       8    4497   6
       UnitDied       1 5846601   1
       UnitDied       2 5782748   2
       UnitDied     NaN 4392174   3
       UnitDied       7    8824   4
       UnitDied       9    5910   5
       UnitDied       0    4529   6
       UnitInit       2 1578447   1


### V2.3 -- join-back validation with neutral/global slicing

Per Amendment 3: match rate computed ONLY on the `player_attributed`
slice. `null_or_missing` and `neutral_or_global` (default `pid IN
(16, 0)`) are reported separately and do NOT count as failures.

In [14]:
v2_3_df = run_q(
    "v2_3_join_back_per_slot_class",
    """
    WITH ev AS (
        SELECT evtTypeName, filename,
            CASE
                WHEN evtTypeName = 'PlayerStats'
                    THEN TRY_CAST(json_extract_string(event_data, '$.playerId') AS INT)
                WHEN evtTypeName IN ('UnitBorn','UnitInit','UnitOwnerChange')
                    THEN TRY_CAST(json_extract_string(event_data, '$.controlPlayerId') AS INT)
                WHEN evtTypeName = 'UnitDied'
                    THEN TRY_CAST(json_extract_string(event_data, '$.killerPlayerId') AS INT)
                WHEN evtTypeName = 'Upgrade'
                    THEN TRY_CAST(json_extract_string(event_data, '$.playerId') AS INT)
                WHEN evtTypeName = 'PlayerSetup'
                    THEN TRY_CAST(json_extract_string(event_data, '$.playerId') AS INT)
            END AS pid
        FROM tracker_events_raw
        WHERE evtTypeName IN (
            'PlayerStats','UnitBorn','UnitInit','UnitOwnerChange',
            'UnitDied','Upgrade','PlayerSetup'
        )
    ),
    classified AS (
        SELECT evtTypeName, filename, pid,
            CASE
                WHEN pid IS NULL THEN 'null_or_missing'
                WHEN pid IN (16, 0) THEN 'neutral_or_global'
                ELSE 'player_attributed'
            END AS slot_class
        FROM ev
    )
    SELECT c.evtTypeName, c.slot_class,
           COUNT(*) AS n_events,
           COUNT(*) FILTER (WHERE rp.toon_id IS NOT NULL) AS n_matched,
           ROUND(
               1.0 * COUNT(*) FILTER (WHERE rp.toon_id IS NOT NULL)
                   / NULLIF(COUNT(*), 0), 6
           ) AS match_rate
    FROM classified c
    LEFT JOIN replay_players_raw rp
        ON rp.filename = c.filename AND rp.playerID = c.pid
    GROUP BY c.evtTypeName, c.slot_class
    ORDER BY c.evtTypeName, c.slot_class
    """,
)
print("=== V2.3 per-(event_type, slot_class) match rate ===")
print(v2_3_df.to_string(index=False))

=== V2.3 per-(event_type, slot_class) match rate ===
    evtTypeName        slot_class  n_events  n_matched  match_rate
    PlayerSetup player_attributed     44817      44817    1.000000
    PlayerStats player_attributed   4558736    4558736    1.000000
       UnitBorn neutral_or_global   3697900          0    0.000000
       UnitBorn player_attributed  18674589   18673868    0.999961
       UnitDied neutral_or_global      4529          0    0.000000
       UnitDied   null_or_missing   4392174          0    0.000000
       UnitDied player_attributed  11657131   11656452    0.999942
       UnitInit player_attributed   3151270    3151270    1.000000
UnitOwnerChange player_attributed     65157      65157    1.000000
        Upgrade neutral_or_global      3636          0    0.000000
        Upgrade player_attributed    794351     794349    0.999997


### V2.4 -- control / upkeep agreement check

For UnitBorn / UnitInit / UnitOwnerChange, both `controlPlayerId`
(true owner) and `upkeepPlayerId` (supply tracking) carry the
slot id. They typically agree but can diverge in mind-control
scenarios. controlPlayerId remains the canonical mapping per
s2protocol snapshot semantics; this cell quantifies the agreement.

In [15]:
v2_4_df = run_q(
    "v2_4_control_upkeep_agreement",
    """
    WITH ev AS (
        SELECT evtTypeName,
            TRY_CAST(json_extract_string(event_data, '$.controlPlayerId') AS INT)
                AS control_pid,
            TRY_CAST(json_extract_string(event_data, '$.upkeepPlayerId') AS INT)
                AS upkeep_pid
        FROM tracker_events_raw
        WHERE evtTypeName IN ('UnitBorn','UnitInit','UnitOwnerChange')
    )
    SELECT evtTypeName,
           COUNT(*) AS n,
           COUNT(*) FILTER (WHERE control_pid = upkeep_pid)
               AS n_agree,
           COUNT(*) FILTER (WHERE control_pid != upkeep_pid)
               AS n_disagree,
           COUNT(*) FILTER (
               WHERE control_pid IS NULL OR upkeep_pid IS NULL
           ) AS n_either_null,
           ROUND(
               1.0 * COUNT(*) FILTER (WHERE control_pid = upkeep_pid)
                   / NULLIF(COUNT(*), 0), 6
           ) AS agreement_rate
    FROM ev
    GROUP BY evtTypeName
    ORDER BY evtTypeName
    """,
)
print("=== V2.4 control vs upkeep agreement per event ===")
print(v2_4_df.to_string(index=False))

=== V2.4 control vs upkeep agreement per event ===
    evtTypeName        n  n_agree  n_disagree  n_either_null  agreement_rate
       UnitBorn 22372489 22372489           0              0        1.000000
       UnitInit  3151270  3151270           0              0        1.000000
UnitOwnerChange    65157    62536        2621              0        0.959774


### V2 verdict assembly

Per-event-type mapping verdict. The 7 direct-mapping events get
`match_rate_player_attributed` from V2.3; the 3 lineage-required
events (UnitTypeChange / UnitDone / UnitPositions) get
`lineage_required` per Amendment-3-aware classification.

In [16]:
DIRECT_MAPPING_FIELD = {
    "PlayerStats": "playerId",
    "UnitBorn": "controlPlayerId",
    "UnitInit": "controlPlayerId",
    "UnitOwnerChange": "controlPlayerId",
    "UnitDied": "killerPlayerId",
    "Upgrade": "playerId",
    "PlayerSetup": "playerId",
}
LINEAGE_REQUIRED = {
    "UnitTypeChange": (
        "No direct player-id field per s2protocol snapshot; "
        "requires lineage via (filename, unitTagIndex, unitTagRecycle) "
        "join back to UnitBorn / UnitInit. Deferred to V5."
    ),
    "UnitDone": (
        "No direct player-id field; lineage via "
        "(filename, unitTagIndex, unitTagRecycle) -> UnitInit. "
        "Deferred to V5."
    ),
    "UnitPositions": (
        "No direct player-id field; uses packed m_items array keyed "
        "by m_firstUnitIndex. Lineage required; coordinate semantics "
        "handled in V6."
    ),
}

### V2 verdict computation

In [17]:
mappings = []
for event_type, field in DIRECT_MAPPING_FIELD.items():
    rows = v2_3_df[v2_3_df["evtTypeName"] == event_type]
    pa = rows[rows["slot_class"] == "player_attributed"]
    ng = rows[rows["slot_class"] == "neutral_or_global"]
    nm = rows[rows["slot_class"] == "null_or_missing"]
    pa_match = float(pa["match_rate"].iloc[0]) if not pa.empty else None
    pa_n = int(pa["n_events"].iloc[0]) if not pa.empty else 0
    ng_n = int(ng["n_events"].iloc[0]) if not ng.empty else 0
    nm_n = int(nm["n_events"].iloc[0]) if not nm.empty else 0
    if pa_match is None:
        confidence = "no_player_attributed_rows"
    elif pa_match >= 0.995:
        confidence = "high"
    elif pa_match >= 0.95:
        confidence = "medium"
    else:
        confidence = "low"
    mappings.append({
        "event_type": event_type,
        "chosen_id_field": field,
        "match_rate_player_attributed": pa_match,
        "n_player_attributed": pa_n,
        "n_neutral_or_global": ng_n,
        "n_null_or_missing": nm_n,
        "confidence": confidence,
        "verdict": (
            "high_confidence_direct_mapping" if confidence == "high"
            else "medium_confidence_direct_mapping" if confidence == "medium"
            else "low_confidence_direct_mapping" if confidence == "low"
            else "no_player_attributed_rows"
        ),
    })
for event_type, reason in LINEAGE_REQUIRED.items():
    mappings.append({
        "event_type": event_type,
        "chosen_id_field": None,
        "match_rate_player_attributed": None,
        "n_player_attributed": None,
        "n_neutral_or_global": None,
        "n_null_or_missing": None,
        "confidence": "n/a",
        "verdict": "lineage_required",
        "rationale": reason,
    })

high_conf = [m for m in mappings if m["verdict"] == "high_confidence_direct_mapping"]
med_conf = [m for m in mappings if m["verdict"] == "medium_confidence_direct_mapping"]
low_or_amb = [m for m in mappings if m["verdict"] in (
    "low_confidence_direct_mapping", "no_player_attributed_rows"
)]
lineage = [m for m in mappings if m["verdict"] == "lineage_required"]

# Determine V2 result.
if low_or_amb:
    v2_result = "FAIL"
    v2_caveat = (
        f"{len(low_or_amb)} direct-mapping event types fell below the "
        f"99.5% / 95% confidence thresholds: "
        f"{[m['event_type'] for m in low_or_amb]}"
    )
elif med_conf:
    v2_result = "PASS_WITH_CAVEAT"
    v2_caveat = (
        f"{len(med_conf)} event types at MEDIUM confidence "
        f"(95% <= match_rate < 99.5%): {[m['event_type'] for m in med_conf]}"
    )
else:
    v2_result = "PASS"
    v2_caveat = (
        f"All {len(high_conf)} direct-mapping event types HIGH confidence; "
        f"{len(lineage)} event types correctly classified lineage_required."
    )

### V2 verdict block

In [18]:
neutral_handling_summary = {
    m["event_type"]: {
        "neutral_or_global_count": m["n_neutral_or_global"],
        "null_or_missing_count": m["n_null_or_missing"],
    }
    for m in mappings
    if m["event_type"] in DIRECT_MAPPING_FIELD
}
control_upkeep_summary = {
    row["evtTypeName"]: {
        "n": int(row["n"]),
        "agreement_rate": float(row["agreement_rate"]),
    }
    for _, row in v2_4_df.iterrows()
}

verdicts["V2"] = {
    "hypothesis": (
        "PlayerStats->playerId; UnitBorn/Init/OwnerChange->controlPlayerId "
        "(+upkeepPlayerId); UnitDied->killerPlayerId (killer attribution; "
        "victim is lineage_required); Upgrade->playerId; PlayerSetup->playerId. "
        "UnitTypeChange/UnitDone/UnitPositions have NO direct player-id "
        "field and require lineage."
    ),
    "falsifier": (
        "no stable mapping field for player-attributed rows; player-attributed "
        "match rate < 99.5% on the player_attributed slice (Amendment 3); "
        "ambiguous ties; direct fields contradict replay_players_raw"
    ),
    "result": v2_result,
    "player_attributed_threshold": 0.995,
    "mappings": mappings,
    "neutral_handling": neutral_handling_summary,
    "control_vs_upkeep_agreement": control_upkeep_summary,
    "ambiguous_event_types": [m["event_type"] for m in low_or_amb],
    "lineage_required_event_types": [m["event_type"] for m in lineage],
    "high_confidence_event_types": [m["event_type"] for m in high_conf],
    "medium_confidence_event_types": [m["event_type"] for m in med_conf],
    "notes": v2_caveat,
}

print("=== V2 verdict ===")
print(f"  result: {verdicts['V2']['result']}")
print(f"  high-confidence direct mappings: "
      f"{verdicts['V2']['high_confidence_event_types']}")
print(f"  medium-confidence direct mappings: "
      f"{verdicts['V2']['medium_confidence_event_types']}")
print(f"  lineage_required: "
      f"{verdicts['V2']['lineage_required_event_types']}")
print(f"  ambiguous/low/fail: "
      f"{verdicts['V2']['ambiguous_event_types']}")
print(f"  notes: {verdicts['V2']['notes']}")

=== V2 verdict ===
  result: PASS
  high-confidence direct mappings: ['PlayerStats', 'UnitBorn', 'UnitInit', 'UnitOwnerChange', 'UnitDied', 'Upgrade', 'PlayerSetup']
  medium-confidence direct mappings: []
  lineage_required: ['UnitTypeChange', 'UnitDone', 'UnitPositions']
  ambiguous/low/fail: []
  notes: All 7 direct-mapping event types HIGH confidence; 3 event types correctly classified lineage_required.


## V2 result summary

- **7 direct-mapping event types:** PlayerStats, UnitBorn, UnitInit,
  UnitOwnerChange, UnitDied, Upgrade, PlayerSetup. Mapping field
  chosen per s2protocol snapshot semantics (controlPlayerId for
  ownership; killerPlayerId for UnitDied attribution; playerId for
  the rest).
- **3 lineage_required event types:** UnitTypeChange, UnitDone,
  UnitPositions. No direct player-id field per s2protocol snapshot;
  classified `lineage_required` (NOT `FAIL`) per Amendment 3
  semantics + plan T07 deferral to V5.
- **Amendment 3 enforced:** match-rate threshold (>=99.5%) applied
  ONLY to the `player_attributed` slice. Neutral/global (pid IN
  (16, 0)) and null/missing rows reported separately, NOT counted as
  failures.

---
## V3 -- PlayerStats field semantics (strict cumulative classification per Q3)

**Hypothesis.**
- PlayerStats records periodic per-player economic / military
  snapshots.
- The stream is suitable for in-game snapshot features if cadence,
  mapping, and field completeness pass.
- Cumulative interpretation is NOT assumed -- the s2protocol decoder
  does not document cumulative semantics for the `*Lost` / `*Killed`
  / `*FriendlyFire` / `*Used` / `*Current` keys. Per Q3 strict rule,
  monotonic empirics ALONE do NOT promote a field to `safe_delta`.

**Falsifier.**
- PlayerStats cadence not stable per `(filename, playerId)`;
- missing PlayerStats for a material number of replays;
- field key-set differs materially from the s2protocol snapshot;
- field values cannot be classified as snapshot / delta / ambiguous;
- any field required for a candidate feature family has unverified
  semantics.

**Classification vocabulary (per T05 step 4).**
- `safe_snapshot` -- per-tick observable, bounded oscillation,
  decreases present (compatible with instantaneous state);
- `safe_delta` -- monotonic non-decreasing AND s2protocol confirms
  cumulative interpretation (Q3 strict requires source confirmation,
  NOT empirical monotonicity alone);
- `unsafe_or_ambiguous` -- neither pattern, OR monotonic-only with
  no source authority.

### V3.1 -- enumerate observed PlayerStats stats keys

In [19]:
v3_1_df = run_q(
    "v3_1_enumerate_stats_keys",
    """
    WITH s AS (
        SELECT json_extract(event_data, '$.stats') AS stats
        FROM tracker_events_raw
        WHERE evtTypeName = 'PlayerStats'
        LIMIT 100000
    ),
    keys_unnested AS (
        SELECT UNNEST(json_keys(stats)) AS k FROM s
    )
    SELECT k, COUNT(*) AS n
    FROM keys_unnested
    GROUP BY k
    ORDER BY k
    """,
)
print("=== V3.1 observed PlayerStats stats keys ===")
print(v3_1_df.to_string(index=False))
observed_keys = sorted(v3_1_df["k"].astype(str).tolist())
print(f"\nObserved key count: {len(observed_keys)}")

=== V3.1 observed PlayerStats stats keys ===
                                         k      n
                        scoreValueFoodMade 100000
                        scoreValueFoodUsed 100000
          scoreValueMineralsCollectionRate 100000
                 scoreValueMineralsCurrent 100000
        scoreValueMineralsFriendlyFireArmy 100000
     scoreValueMineralsFriendlyFireEconomy 100000
  scoreValueMineralsFriendlyFireTechnology 100000
              scoreValueMineralsKilledArmy 100000
           scoreValueMineralsKilledEconomy 100000
        scoreValueMineralsKilledTechnology 100000
                scoreValueMineralsLostArmy 100000
             scoreValueMineralsLostEconomy 100000
          scoreValueMineralsLostTechnology 100000
        scoreValueMineralsUsedActiveForces 100000
         scoreValueMineralsUsedCurrentArmy 100000
      scoreValueMineralsUsedCurrentEconomy 100000
   scoreValueMineralsUsedCurrentTechnology 100000
      scoreValueMineralsUsedInProgressArmy 100000
   sc

### V3.1 -- cross-reference observed vs s2protocol snapshot

**Empirical convention discovered.** The s2protocol snapshot uses the
C++ struct member prefix `m_` (e.g., `m_scoreValueMineralsCurrent`).
The SC2EGSet JSON decoder STRIPS this prefix when serializing event
payloads, so the observed JSON keys are `scoreValueMineralsCurrent`
(no leading `m_`). The set comparison normalizes both sides by
stripping `m_` from the snapshot keys.

In [20]:
expected_keys_raw = sorted(
    EVIDENCE["V3_player_stats_fields"]["stats_keys_from_snapshot"]
)
expected_keys = sorted(k.removeprefix("m_") for k in expected_keys_raw)
observed_set = set(observed_keys)
expected_set = set(expected_keys)
missing_from_observed = sorted(expected_set - observed_set)
extra_in_observed = sorted(observed_set - expected_set)
matched_set = sorted(observed_set & expected_set)
print(f"observed_count: {len(observed_keys)}")
print(f"expected_count (m_-stripped): {len(expected_keys)}")
print(f"matched_count: {len(matched_set)}")
print(f"missing_from_observed ({len(missing_from_observed)}): "
      f"{missing_from_observed}")
print(f"extra_in_observed   ({len(extra_in_observed)}): "
      f"{extra_in_observed}")
# Per T05: do not fail if snapshot has keys absent historically; record
# for V4 if needed.
EVIDENCE["V3_player_stats_fields"]["decoded_json_strips_m_prefix"] = True

observed_count: 39
expected_count (m_-stripped): 39
matched_count: 39
missing_from_observed (0): []
extra_in_observed   (0): []


### V3.2 -- cadence audit partitioned by (filename, playerId)

Top gaps in the per-player tracker series. Per-player partitioning
eliminates the cross-player co-fire seen in `research_log.md` lines
~991-994. Any `gap=0` rows that remain are NOT cross-player co-fire
(the partition handles that) -- they are duplicate PlayerStats rows
for the same `(filename, playerId, loop)`. The dedicated cell below
verifies this empirically: the count of `(extra rows beyond the
first per duplicate group)` should match the `gap=0` row count.

In [21]:
v3_2_gap_df = run_q(
    "v3_2_cadence_gap_distribution",
    """
    WITH ps AS (
        SELECT filename,
               TRY_CAST(json_extract_string(event_data, '$.playerId') AS INT)
                   AS pid,
               loop,
               loop - LAG(loop) OVER (
                   PARTITION BY
                       filename,
                       TRY_CAST(json_extract_string(event_data, '$.playerId')
                                AS INT)
                   ORDER BY loop
               ) AS gap
        FROM tracker_events_raw
        WHERE evtTypeName = 'PlayerStats'
    )
    SELECT gap, COUNT(*) AS n
    FROM ps
    WHERE gap IS NOT NULL
    GROUP BY gap
    ORDER BY n DESC
    LIMIT 10
    """,
)
print("=== V3.2 top-10 gap distribution per (filename, playerId) ===")
print(v3_2_gap_df.to_string(index=False))

=== V3.2 top-10 gap distribution per (filename, playerId) ===
 gap       n
 160 4401952
 159   45036
   0   22662
  43     353
  95     340
 119     332
  17     332
   7     331
  74     326
  90     319


### V3.2 -- per-replay coverage and per-player presence

In [22]:
v3_2_coverage_df = run_q(
    "v3_2_per_replay_player_coverage",
    """
    WITH ps_player AS (
        SELECT filename,
               TRY_CAST(json_extract_string(event_data, '$.playerId') AS INT)
                   AS pid
        FROM tracker_events_raw
        WHERE evtTypeName = 'PlayerStats'
        GROUP BY filename, pid
    ),
    per_replay AS (
        SELECT filename, COUNT(DISTINCT pid) AS n_distinct_players
        FROM ps_player
        GROUP BY filename
    )
    SELECT n_distinct_players, COUNT(*) AS n_replays
    FROM per_replay
    GROUP BY n_distinct_players
    ORDER BY n_distinct_players
    """,
)
print("=== V3.2 distinct PlayerStats playerIds per replay ===")
print(v3_2_coverage_df.to_string(index=False))

=== V3.2 distinct PlayerStats playerIds per replay ===
 n_distinct_players  n_replays
                  1          3
                  2      22379
                  4          2
                  6          1
                  8          3
                  9          2


### V3.2 -- verdict assembly

In [23]:
total_gap_rows = int(v3_2_gap_df["n"].sum())
dominant_gap_row = v3_2_gap_df.iloc[0]
dominant_gap = int(dominant_gap_row["gap"])
dominant_gap_n = int(dominant_gap_row["n"])
dominant_gap_frac = (
    dominant_gap_n / total_gap_rows if total_gap_rows else 0.0
)
top_10_gaps = [
    {"gap": int(r["gap"]), "n": int(r["n"])}
    for _, r in v3_2_gap_df.iterrows()
]
n_replays_with_both_players = int(
    v3_2_coverage_df[v3_2_coverage_df["n_distinct_players"] == 2]
    ["n_replays"].sum()
)
n_replays_total_in_v32 = int(v3_2_coverage_df["n_replays"].sum())
n_replay_player_pairs_no_ps = 0
for _, r in v3_2_coverage_df.iterrows():
    if int(r["n_distinct_players"]) < 2:
        n_replay_player_pairs_no_ps += int(r["n_replays"]) * (
            2 - int(r["n_distinct_players"])
        )
cadence_pass_160 = (dominant_gap == 160) and (dominant_gap_frac >= 0.95)
print(
    f"dominant_gap={dominant_gap}, frac={dominant_gap_frac:.4f}, "
    f"replays_with_both_players={n_replays_with_both_players}/"
    f"{n_replays_total_in_v32}, missing_replay_player_pairs="
    f"{n_replay_player_pairs_no_ps}, cadence_pass={cadence_pass_160}"
)

dominant_gap=160, frac=0.9843, replays_with_both_players=22379/22390, missing_replay_player_pairs=3, cadence_pass=True


### V3.2 -- explain gap=0: duplicate-row vs cross-player co-fire

The partition on `(filename, playerId)` already excludes cross-player
co-fire from the gap series. A non-zero `gap=0` count therefore
implies same-player duplicate PlayerStats rows: multiple rows with
identical `(filename, playerId, loop)`. The query below counts those
duplicates; the sum of `n - 1` per duplicate group must equal the
`gap=0` row count from V3.2.

In [24]:
v3_2_dup_df = run_q(
    "v3_2_same_player_duplicate_groups",
    """
    WITH ps AS (
        SELECT filename,
               TRY_CAST(json_extract_string(event_data, '$.playerId') AS INT)
                   AS pid,
               loop,
               COUNT(*) AS n_at_same_key
        FROM tracker_events_raw
        WHERE evtTypeName = 'PlayerStats'
        GROUP BY filename, pid, loop
    )
    SELECT
        COUNT(*) FILTER (WHERE n_at_same_key > 1) AS n_duplicate_groups,
        COALESCE(SUM(n_at_same_key - 1) FILTER (WHERE n_at_same_key > 1), 0)
            AS n_extra_rows_beyond_first
    FROM ps
    """,
)
n_duplicate_groups = int(v3_2_dup_df["n_duplicate_groups"].iloc[0])
n_extra_rows = int(v3_2_dup_df["n_extra_rows_beyond_first"].iloc[0])
gap_zero_row = v3_2_gap_df[v3_2_gap_df["gap"] == 0]
gap_zero_n = int(gap_zero_row["n"].iloc[0]) if not gap_zero_row.empty else 0
gap_zero_explained_by_duplicates = (n_extra_rows == gap_zero_n)
print(f"duplicate_groups={n_duplicate_groups}, "
      f"extra_rows_beyond_first={n_extra_rows}, "
      f"gap_zero_n={gap_zero_n}, "
      f"explained_by_duplicates={gap_zero_explained_by_duplicates}")

duplicate_groups=22512, extra_rows_beyond_first=22662, gap_zero_n=22662, explained_by_duplicates=True


### V3.3 -- field classification (per-(filename, playerId) monotonicity)

Single SQL pass extracts every observed key as a DOUBLE column,
computes LAG within (filename, playerId) ordered by loop, then
aggregates: null count, min, max, n_with_prev, n_decreases. Per-field
null fractions and decrease fractions are computed in Python.

In [25]:
def _build_field_stats_sql(keys: list[str]) -> str:
    """Build a single-pass SQL that aggregates monotonicity stats per key."""
    extracts = ",\n        ".join(
        f"TRY_CAST(json_extract_string(event_data, '$.stats.{k}') "
        f"AS DOUBLE) AS \"{k}\""
        for k in keys
    )
    lags = ",\n        ".join(
        f'"{k}", LAG("{k}") OVER w AS "prev_{k}"' for k in keys
    )
    aggs = ",\n  ".join(
        f'SUM(CASE WHEN "{k}" IS NULL THEN 1 ELSE 0 END) AS "n_null_{k}", '
        f'MIN("{k}") AS "min_{k}", MAX("{k}") AS "max_{k}", '
        f'SUM(CASE WHEN "prev_{k}" IS NOT NULL THEN 1 ELSE 0 END) '
        f'AS "n_wp_{k}", '
        f'SUM(CASE WHEN "prev_{k}" IS NOT NULL AND "{k}" < "prev_{k}" '
        f'THEN 1 ELSE 0 END) AS "n_dec_{k}"'
        for k in keys
    )
    return (
        "WITH ps AS (\n"
        "  SELECT filename,\n"
        "    TRY_CAST(json_extract_string(event_data, '$.playerId') AS INT)\n"
        "      AS pid,\n"
        "    loop,\n"
        f"    {extracts}\n"
        "  FROM tracker_events_raw\n"
        "  WHERE evtTypeName = 'PlayerStats'\n"
        "),\n"
        "with_lag AS (\n"
        f"  SELECT filename, pid, loop,\n        {lags}\n"
        "  FROM ps\n"
        "  WINDOW w AS (PARTITION BY filename, pid ORDER BY loop)\n"
        ")\n"
        f"SELECT COUNT(*) AS n_total_rows,\n  {aggs}\nFROM with_lag"
    )

In [26]:
v3_3_sql = _build_field_stats_sql(observed_keys)
sql_queries["v3_3_field_stats"] = v3_3_sql
print(f"V3.3 SQL length: {len(v3_3_sql)} chars; "
      f"keys: {len(observed_keys)}")
v3_3_df = conn.con.execute(v3_3_sql).df()
n_total_rows = int(v3_3_df["n_total_rows"].iloc[0])
print(f"V3.3 scanned n_total_rows = {n_total_rows}")

V3.3 SQL length: 35956 chars; keys: 39


V3.3 scanned n_total_rows = 4558736


### V3.3 -- per-field summary stats

In [27]:
import pandas as pd

field_stats: list[dict] = []
for k in observed_keys:
    n_null = int(v3_3_df[f"n_null_{k}"].iloc[0])
    minv = v3_3_df[f"min_{k}"].iloc[0]
    maxv = v3_3_df[f"max_{k}"].iloc[0]
    n_wp = int(v3_3_df[f"n_wp_{k}"].iloc[0])
    n_dec = int(v3_3_df[f"n_dec_{k}"].iloc[0])
    null_frac = n_null / n_total_rows if n_total_rows else None
    dec_frac = n_dec / n_wp if n_wp else None
    field_stats.append({
        "key": k,
        "n_null": n_null,
        "null_fraction": null_frac,
        "min": float(minv) if minv is not None else None,
        "max": float(maxv) if maxv is not None else None,
        "n_with_prev": n_wp,
        "n_decreases": n_dec,
        "frac_decreases": dec_frac,
    })
fs_df = pd.DataFrame(field_stats)
print("=== V3.3 per-field summary statistics ===")
print(
    fs_df[["key", "null_fraction", "min", "max", "frac_decreases"]]
    .to_string(index=False)
)

=== V3.3 per-field summary statistics ===
                                       key  null_fraction     min     max  frac_decreases
                        scoreValueFoodMade            0.0     0.0   573.0        0.025038
                        scoreValueFoodUsed            0.0     0.0   213.0        0.177641
          scoreValueMineralsCollectionRate            0.0     0.0  6187.0        0.349859
                 scoreValueMineralsCurrent            0.0     0.0 23641.0        0.420318
        scoreValueMineralsFriendlyFireArmy            0.0     0.0  9050.0        0.000000
     scoreValueMineralsFriendlyFireEconomy            0.0     0.0  2950.0        0.000000
  scoreValueMineralsFriendlyFireTechnology            0.0     0.0 12725.0        0.000000
              scoreValueMineralsKilledArmy            0.0     0.0 65650.0        0.000000
           scoreValueMineralsKilledEconomy            0.0     0.0 23300.0        0.000000
        scoreValueMineralsKilledTechnology            0.0 

### V3.3 -- apply Q3 strict classification rule

Per Q3 + T02 EVIDENCE: s2protocol does NOT document cumulative
semantics for any `*Lost` / `*Killed` / `*FriendlyFire` / `*Used` /
`*Current` field. Therefore monotonic empirics ALONE cannot promote a
field to `safe_delta`. Monotonic-only fields default to
`unsafe_or_ambiguous`. The empty `SOURCE_CONFIRMS_CUMULATIVE` set
encodes this explicitly.

In [28]:
SOURCE_CONFIRMS_CUMULATIVE: set[str] = set()
classified: list[dict] = []
for fs in field_stats:
    k = fs["key"]
    dec_frac = fs["frac_decreases"]
    if dec_frac is None or fs["n_with_prev"] == 0:
        cls, reason = "unsafe_or_ambiguous", "insufficient_consecutive_pairs"
    elif fs["min"] is not None and fs["min"] == fs["max"]:
        cls = "unsafe_or_ambiguous"
        reason = (
            f"constant value {fs['min']}; cannot distinguish snapshot "
            "vs cumulative"
        )
    elif dec_frac > 0:
        cls = "safe_snapshot"
        reason = f"bounded oscillation; frac_decreases={dec_frac:.6f}"
    elif k in SOURCE_CONFIRMS_CUMULATIVE:
        cls = "safe_delta"
        reason = "monotonic non-decreasing AND s2protocol cumulative-confirmed"
    else:
        cls = "unsafe_or_ambiguous"
        reason = (
            f"monotonic non-decreasing (frac_decreases={dec_frac:.6f}) "
            "but s2protocol silent on cumulative semantics (Q3 strict)"
        )
    classified.append({
        **fs,
        "classification": cls,
        "classification_reason": reason,
        "source_confirmed_cumulative": k in SOURCE_CONFIRMS_CUMULATIVE,
    })

import collections
class_counts = collections.Counter(c["classification"] for c in classified)
print("=== V3.3 classification counts ===")
print(dict(class_counts))

=== V3.3 classification counts ===
{'safe_snapshot': 26, 'unsafe_or_ambiguous': 13}


### V3.3 -- fixed-point handling note

Per s2protocol README, `m_scoreValueFoodUsed` and `m_scoreValueFoodMade`
are stored in fixed point (divide by 4096 for integer values).
Whether the SC2EGSet decoded JSON is pre-scaled is NOT confirmed by
this notebook. T05 records the convention verbatim and does NOT apply
any divide-by-4096 transformation. Phase 02 must resolve scaling
discipline before any feature uses these fields.

In [29]:
FIXED_POINT_KEYS_SNAPSHOT = ["m_scoreValueFoodUsed", "m_scoreValueFoodMade"]
FIXED_POINT_KEYS_OBSERVED = [
    k.removeprefix("m_") for k in FIXED_POINT_KEYS_SNAPSHOT
]
fixed_point_handling = {
    "policy": (
        "T05 records fixed-point convention from s2protocol README "
        "verbatim; does NOT divide by 4096; classification proceeds on "
        "raw decoded values; Phase 02 must resolve scaling discipline."
    ),
    "verbatim_note": (
        EVIDENCE["V3_player_stats_fields"]
        ["s2protocol_fixed_point_note_verbatim"]
    ),
    "fields_in_snapshot": FIXED_POINT_KEYS_SNAPSHOT,
    "fields_in_observed_form": FIXED_POINT_KEYS_OBSERVED,
    "observed_in_sc2egset": [
        k for k in FIXED_POINT_KEYS_OBSERVED if k in observed_keys
    ],
}
print("=== V3.3 fixed-point handling ===")
for fk, fv in fixed_point_handling.items():
    print(f"  {fk}: {fv}")

=== V3.3 fixed-point handling ===
  policy: T05 records fixed-point convention from s2protocol README verbatim; does NOT divide by 4096; classification proceeds on raw decoded values; Phase 02 must resolve scaling discipline.
  verbatim_note: m_scoreValueFoodUsed and m_scoreValueFoodMade are in fixed point (divide by 4096 for integer values). All other values are in integers.
  fields_in_snapshot: ['m_scoreValueFoodUsed', 'm_scoreValueFoodMade']
  fields_in_observed_form: ['scoreValueFoodUsed', 'scoreValueFoodMade']
  observed_in_sc2egset: ['scoreValueFoodUsed', 'scoreValueFoodMade']


### V3 verdict assembly

`cumulative_economy_blocked` is True iff at least one
`*Lost` / `*Killed` / `*FriendlyFire` field is NOT classified
`safe_delta`. Per the empty `SOURCE_CONFIRMS_CUMULATIVE` set, this is
True by construction in T05.

In [30]:
safe_snapshot_fields = sorted(
    c["key"] for c in classified if c["classification"] == "safe_snapshot"
)
safe_delta_fields = sorted(
    c["key"] for c in classified if c["classification"] == "safe_delta"
)
unsafe_fields = sorted(
    c["key"] for c in classified
    if c["classification"] == "unsafe_or_ambiguous"
)
all_classified = (len(classified) == len(observed_keys))
cum_kw = ("Lost", "Killed", "FriendlyFire")
cumulative_economy_blocked = any(
    c["classification"] != "safe_delta"
    for c in classified
    if any(w in c["key"] for w in cum_kw)
)

if not cadence_pass_160:
    v3_result = "FAIL"
    v3_caveat = (
        f"cadence audit: dominant_gap={dominant_gap} "
        f"(frac={dominant_gap_frac:.4f}); expected 160 with frac>=0.95"
    )
elif missing_from_observed or extra_in_observed:
    v3_result = "PASS_WITH_CAVEAT"
    v3_caveat = (
        f"key set drift vs s2protocol snapshot: "
        f"missing={len(missing_from_observed)}, "
        f"extra={len(extra_in_observed)}; cumulative-economy fields "
        f"blocked per Q3 strict"
    )
else:
    v3_result = "PASS_WITH_CAVEAT"
    v3_caveat = (
        "snapshot cadence and key-set match; "
        f"{len(safe_snapshot_fields)} safe_snapshot fields enable "
        "in-game snapshot features; cumulative-economy features "
        "blocked because s2protocol silent on cumulative semantics "
        "(Q3 strict)"
    )

### V3 verdict block

In [31]:
verdicts["V3"] = {
    "hypothesis": (
        "PlayerStats records periodic per-player economic / military "
        "snapshots; suitable for in-game snapshot features if cadence, "
        "mapping, and field completeness pass; cumulative interpretation "
        "NOT assumed (Q3 strict)."
    ),
    "falsifier": (
        "cadence not stable per (filename, playerId); missing PlayerStats "
        "for material number of replays; key-set drift; values cannot be "
        "classified; candidate feature family field semantics unverified"
    ),
    "result": v3_result,
    "all_observed_keys_classified": all_classified,
    "cadence_summary": {
        "dominant_gap_loops": dominant_gap,
        "dominant_gap_fraction": dominant_gap_frac,
        "top_10_gaps": top_10_gaps,
        "replays_with_both_players": n_replays_with_both_players,
        "replays_total_with_playerstats": n_replays_total_in_v32,
        "missing_replay_player_pairs": n_replay_player_pairs_no_ps,
        "expected_dominant_gap_loops": 160,
        "expected_dominant_gap_seconds_at_22_4_lps": 160 / 22.4,
        "cadence_pass_160": cadence_pass_160,
        "gap_zero_n": gap_zero_n,
        "gap_zero_cause": (
            "same-player duplicate PlayerStats tick rows; partition by "
            "(filename, playerId) is correct"
        ),
        "gap_zero_explained_by_duplicates": gap_zero_explained_by_duplicates,
        "duplicate_groups": n_duplicate_groups,
        "duplicate_extra_rows_beyond_first": n_extra_rows,
    },
    "key_set_comparison": {
        "observed_count": len(observed_keys),
        "expected_count": len(expected_keys),
        "matched_count": len(matched_set),
        "missing_from_observed": missing_from_observed,
        "extra_in_observed": extra_in_observed,
    },
    "field_classification": classified,
    "safe_snapshot_fields": safe_snapshot_fields,
    "safe_delta_fields": safe_delta_fields,
    "unsafe_or_ambiguous_fields": unsafe_fields,
    "cumulative_economy_blocked": cumulative_economy_blocked,
    "fixed_point_handling": fixed_point_handling,
    "notes": v3_caveat,
}
print("=== V3 verdict ===")
print(f"  result: {verdicts['V3']['result']}")
print(f"  safe_snapshot fields: {len(safe_snapshot_fields)}")
print(f"  safe_delta fields:    {len(safe_delta_fields)}")
print(f"  unsafe_or_ambiguous:  {len(unsafe_fields)}")
print(f"  cumulative_economy_blocked: {cumulative_economy_blocked}")
print(f"  notes: {verdicts['V3']['notes']}")

=== V3 verdict ===
  result: PASS_WITH_CAVEAT
  safe_snapshot fields: 26
  safe_delta fields:    0
  unsafe_or_ambiguous:  13
  cumulative_economy_blocked: True
  notes: snapshot cadence and key-set match; 26 safe_snapshot fields enable in-game snapshot features; cumulative-economy features blocked because s2protocol silent on cumulative semantics (Q3 strict)


### V3 classification consistency assertions

Per T05 hygiene check section B: assert that the classification is
total, mutually exclusive, and that `safe_delta` remains 0 (since
`SOURCE_CONFIRMS_CUMULATIVE` is empty by design under Q3 strict).

In [32]:
assert len(classified) == len(observed_keys), (
    f"classified count {len(classified)} != observed {len(observed_keys)}"
)
class_buckets = {c["key"]: c["classification"] for c in classified}
assert len(class_buckets) == len(classified), "duplicate keys in classified"
assert set(class_buckets.values()) <= {
    "safe_snapshot", "safe_delta", "unsafe_or_ambiguous"
}, "unexpected classification value"
ss = set(safe_snapshot_fields)
sd = set(safe_delta_fields)
ua = set(unsafe_fields)
assert (ss | sd | ua) == set(observed_keys), (
    f"union of class lists != observed keys; "
    f"diff={(ss | sd | ua) ^ set(observed_keys)}"
)
assert (ss & sd) == set(), f"safe_snapshot ∩ safe_delta = {ss & sd}"
assert (ss & ua) == set(), f"safe_snapshot ∩ unsafe = {ss & ua}"
assert (sd & ua) == set(), f"safe_delta ∩ unsafe = {sd & ua}"
assert len(safe_delta_fields) == 0, (
    "safe_delta MUST be empty under Q3 strict: SOURCE_CONFIRMS_CUMULATIVE "
    "is empty by design"
)
print("=== V3 classification consistency: OK ===")
print(f"  total: {len(classified)}; "
      f"safe_snapshot: {len(safe_snapshot_fields)}; "
      f"safe_delta: {len(safe_delta_fields)}; "
      f"unsafe_or_ambiguous: {len(unsafe_fields)}")

=== V3 classification consistency: OK ===
  total: 39; safe_snapshot: 26; safe_delta: 0; unsafe_or_ambiguous: 13


## V3 result summary

- **Cadence:** dominant gap is 160 loops per `(filename, playerId)`,
  matching the SC2 tracker spec (~7.14s at 22.4 lps). The residual
  `gap=0` rows are NOT cross-player co-fire (partition handles that)
  -- they are same-player duplicate PlayerStats rows for the same
  `(filename, playerId, loop)`. Row count exactly matches the
  `n - 1` sum over duplicate groups, so this is a row-multiplicity
  caveat at ingestion, not a cadence partition error.
- **Key set:** observed PlayerStats stats keys match the s2protocol
  `protocol88500.py` snapshot after stripping the `m_` prefix (the
  SC2EGSet decoder strips that C++ struct member prefix when
  serializing JSON).
- **`safe_snapshot` (26 fields)** -- usable as a raw observed snapshot
  at a cutoff loop, with scaling and semantic caveats where flagged
  below. NOT a cumulative total. Includes:
    - all `*Current` / `*UsedCurrent*` / `*UsedInProgress*` /
      `*UsedActiveForces` / `*CollectionRate` /
      `WorkersActiveCount` / `FoodUsed` / `FoodMade`
      (oscillation observed, as expected for instantaneous state);
    - 5 of 6 `*Lost*` fields (`scoreValueMineralsLostArmy/Economy/
      Technology`, `scoreValueVespeneLostArmy`,
      `scoreValueVespeneLostTechnology`) -- small empirical decrease
      fractions and negative minima (e.g., min=-2033) suggest these
      are not pure cumulative tallies; treated as raw snapshots with
      a "negative-min caveat" for Phase 02.
- **`unsafe_or_ambiguous` (13 fields)** -- monotonic non-decreasing
  AND s2protocol silent on cumulative semantics, OR constant-zero:
    - all 6 `*Killed*` fields (Mineral and Vespene, Army/Economy/
      Technology) -- strictly monotonic, no source confirmation;
    - all 6 `*FriendlyFire*` fields -- mostly constant zero in 1v1
      Random Map, no source confirmation;
    - 1 of 6 `*Lost*` fields (`scoreValueVespeneLostEconomy`) --
      zero decreases observed AND no source confirmation.
- **`safe_delta` (0 fields)** -- empty by construction. Under Q3
  strict, `SOURCE_CONFIRMS_CUMULATIVE` is empty: monotonic empirics
  alone CANNOT promote a field to `safe_delta` without s2protocol
  confirmation, which is absent for every PlayerStats stats key.
- **Cumulative-economy feature families are blocked** until additional
  validation (Q3 strict + Amendment 1 aggregation in T10);
  snapshot feature families remain candidates pending V4..V7.
- **`safe_snapshot` does NOT mean cumulative total.** It means
  "usable as a raw observed snapshot at a cutoff loop, subject to the
  scaling caveat (FoodUsed/FoodMade) and the negative-minimum caveat
  (Lost fields)." Cumulative-style features remain blocked.
- **V3 verdict:** `PASS_WITH_CAVEAT` -- snapshot cadence and key-set
  pass; cumulative-economy feature families blocked per Q3 strict.

---
## V4 -- Event coverage and JSON key-set stability across years / patches

**Hypothesis.** All 10 tracker event families observed in 01_03_04
have stable enough presence and JSON key structure across the
2016-2024 corpus to support feature-family eligibility decisions.

**Falsifier.**
- any event family used by a planned feature is missing or materially
  unstable across major year / `gameVersion` cohorts;
- key-set changes materially across cohorts (>=2 keys differ between
  two cohorts each holding >5% of the corpus) and cannot be handled
  by source-specific null/caveat logic;
- rare-event under-sampling prevents a defensible decision.

**Carry-forward (T02).** `protocol88500.py` is a *recent* s2protocol
reference snapshot; V4 must empirically check observed corpus
stability across the 2016-2024 SC2EGSet window. Do NOT claim that
protocol88500 alone proves full historical stability.

**Per Amendment 6.** Rare event families MUST NOT be declared
absent/unstable solely because Pass A 1% Bernoulli sampling
under-represents them. Pass B stratified resample (up to 10K rows per
`(evtTypeName, cohort)`) confirms or refutes Pass A findings; truly
rare families get `coverage_too_sparse_for_stability_decision`.

### V4.1 -- per-year replay coverage and event count (full table)

Joins `tracker_events_raw` x `replays_meta_raw` on `filename` and
extracts the year from `details.timeUTC` via `TRY_CAST`. Full-table
scan -- no sampling here.

In [33]:
v4_1_year_totals_df = run_q(
    "v4_1_year_replay_totals",
    """
    SELECT EXTRACT(YEAR FROM TRY_CAST(details.timeUTC AS TIMESTAMP)) AS year,
           COUNT(*) AS n_replays_in_year
    FROM replays_meta_raw
    GROUP BY year
    ORDER BY year
    """,
)
print("=== V4.1 replays per year ===")
print(v4_1_year_totals_df.to_string(index=False))
years_observed = sorted(
    int(y) for y in v4_1_year_totals_df["year"].dropna().tolist()
)
print(f"\nObserved years: {years_observed}")

=== V4.1 replays per year ===
 year  n_replays_in_year
 2016                555
 2017               1999
 2018               3180
 2019               3886
 2020               2842
 2021               3836
 2022               3294
 2023               1752
 2024               1046

Observed years: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]


In [34]:
v4_1_df = run_q(
    "v4_1_per_year_coverage",
    """
    WITH years AS (
        SELECT DISTINCT
               EXTRACT(YEAR FROM TRY_CAST(details.timeUTC AS TIMESTAMP)) AS year
        FROM replays_meta_raw
        WHERE TRY_CAST(details.timeUTC AS TIMESTAMP) IS NOT NULL
    ),
    event_types AS (
        SELECT DISTINCT evtTypeName FROM tracker_events_raw
    ),
    grid AS (
        SELECT y.year, et.evtTypeName
        FROM years y CROSS JOIN event_types et
    ),
    counts AS (
        SELECT EXTRACT(YEAR FROM TRY_CAST(rm.details.timeUTC AS TIMESTAMP))
                   AS year,
               te.evtTypeName,
               COUNT(DISTINCT te.filename) AS n_replays,
               COUNT(*) AS n_events
        FROM tracker_events_raw te
        JOIN replays_meta_raw rm USING (filename)
        GROUP BY year, te.evtTypeName
    )
    SELECT g.year, g.evtTypeName,
           COALESCE(c.n_replays, 0) AS n_replays,
           COALESCE(c.n_events, 0) AS n_events
    FROM grid g
    LEFT JOIN counts c USING (year, evtTypeName)
    ORDER BY g.year, g.evtTypeName
    """,
)
v4_1_with_pct = v4_1_df.merge(
    v4_1_year_totals_df, on="year", how="left"
)
v4_1_with_pct["coverage_pct"] = (
    v4_1_with_pct["n_replays"] / v4_1_with_pct["n_replays_in_year"]
)
print("=== V4.1 per-(year, evtTypeName) coverage (zero rows included) ===")
print(v4_1_with_pct.to_string(index=False))

=== V4.1 per-(year, evtTypeName) coverage (zero rows included) ===
 year     evtTypeName  n_replays  n_events  n_replays_in_year  coverage_pct
 2016     PlayerSetup        555      1110                555      1.000000
 2016     PlayerStats        555    104495                555      1.000000
 2016        UnitBorn        555    467914                555      1.000000
 2016        UnitDied        554    292961                555      0.998198
 2016        UnitDone        554     52781                555      0.998198
 2016        UnitInit        554     55105                555      0.998198
 2016 UnitOwnerChange          0         0                555      0.000000
 2016   UnitPositions        553     20906                555      0.996396
 2016  UnitTypeChange        554    268240                555      0.998198
 2016         Upgrade        550     12229                555      0.990991
 2017     PlayerSetup       1999      4004               1999      1.000000
 2017     PlayerStats

### V4.1 -- coverage stability per event type (across years)

Stability classification (per "don't overfail rare families"):
- "presence" = year has >=1 replay containing the event family.
- present-year spread is computed over years with presence ONLY (not
  over zero rows -- absence is a separate dimension).
- high-prevalence family (max present-year cov >= 0.9): stable iff
  *absolute* spread <= 5%;
- sparse family (max present-year cov < 0.9): stable iff *relative*
  spread <= 30%;
- sparse + absent in some years (e.g., UnitOwnerChange absent in 2016):
  `coverage_stable_by_year=True` IF spread is within the family's
  threshold AND presence covers >= 70% of observed years; a
  `coverage_caveat` is recorded and a `feature_eligibility_caveat`
  propagates the sparse/absent fact to V8.
- absent in majority of years: `too_sparse_to_decide`.
`stability_basis` records which rule applied so the verdict is auditable.

In [35]:
event_types_observed = sorted(v4_1_df["evtTypeName"].unique().tolist())
years_total_set = set(int(y) for y in v4_1_year_totals_df["year"].dropna())
year_coverage_summary: dict = {}
for et in event_types_observed:
    rows = v4_1_with_pct[v4_1_with_pct["evtTypeName"] == et]
    rows_present = rows[rows["n_replays"] > 0]
    years_present = sorted(
        int(y) for y in rows_present["year"].dropna().tolist()
    )
    years_absent = sorted(years_total_set - set(years_present))
    present_pcts = rows_present["coverage_pct"].tolist()
    if not present_pcts:
        year_coverage_summary[et] = {
            "years_present": [], "years_absent": sorted(years_total_set),
            "min_year_coverage": None, "max_year_coverage": None,
            "coverage_spread": None, "coverage_relative_spread": None,
            "stability_basis": "no_presence_in_any_year",
            "coverage_stable_by_year": "too_sparse_to_decide",
            "coverage_caveat": "no presence in any observed year",
            "feature_eligibility_caveat":
                "sparse_event_family_not_broadly_available",
        }
        continue
    minp = float(min(present_pcts)); maxp = float(max(present_pcts))
    spread = maxp - minp
    rel_spread = spread / maxp if maxp > 0 else 0.0
    if maxp >= 0.9:
        spread_basis = "absolute_spread<=0.05 (high-prevalence)"
        spread_stable = (spread <= 0.05)
    else:
        spread_basis = "relative_spread<=0.30 (sparse-family)"
        spread_stable = (rel_spread <= 0.30)
    if not years_absent:
        cls = spread_stable
        basis = spread_basis
        cov_caveat = None
        feat_caveat = None
    elif len(years_present) >= 0.7 * len(years_total_set):
        cls = True if spread_stable else False
        basis = (
            f"{spread_basis} + sparse-family caveat: absent in "
            f"{years_absent}"
        )
        cov_caveat = (
            f"sparse / absent in years {years_absent} but present in "
            f"{len(years_present)} of {len(years_total_set)} years; "
            f"present-year spread within family threshold"
        )
        feat_caveat = "sparse_event_family_not_broadly_available"
    else:
        cls = "too_sparse_to_decide"
        basis = "absent_in_majority_of_years"
        cov_caveat = (
            f"absent in {len(years_absent)} of {len(years_total_set)} "
            f"years -> insufficient year coverage to decide stability"
        )
        feat_caveat = "sparse_event_family_not_broadly_available"
    year_coverage_summary[et] = {
        "years_present": years_present,
        "years_absent": years_absent,
        "min_year_coverage": minp,
        "max_year_coverage": maxp,
        "coverage_spread": spread,
        "coverage_relative_spread": rel_spread,
        "stability_basis": basis,
        "coverage_stable_by_year": cls,
        "coverage_caveat": cov_caveat,
        "feature_eligibility_caveat": feat_caveat,
    }
print("=== V4.1 per-event year coverage summary ===")
for et in event_types_observed:
    s = year_coverage_summary[et]
    print(
        f"  {et}: present={len(s['years_present'])}, "
        f"absent={s['years_absent']}, "
        f"min={s['min_year_coverage']:.4f}, "
        f"max={s['max_year_coverage']:.4f}, "
        f"abs_spread={s['coverage_spread']:.4f}, "
        f"rel_spread={s['coverage_relative_spread']:.4f}, "
        f"stable={s['coverage_stable_by_year']}"
    )
    if s["coverage_caveat"]:
        print(f"    caveat: {s['coverage_caveat']}")

=== V4.1 per-event year coverage summary ===
  PlayerSetup: present=9, absent=[], min=1.0000, max=1.0000, abs_spread=0.0000, rel_spread=0.0000, stable=True
  PlayerStats: present=9, absent=[], min=1.0000, max=1.0000, abs_spread=0.0000, rel_spread=0.0000, stable=True
  UnitBorn: present=9, absent=[], min=1.0000, max=1.0000, abs_spread=0.0000, rel_spread=0.0000, stable=True
  UnitDied: present=9, absent=[], min=0.9920, max=0.9997, abs_spread=0.0077, rel_spread=0.0077, stable=True
  UnitDone: present=9, absent=[], min=0.9920, max=1.0000, abs_spread=0.0080, rel_spread=0.0080, stable=True
  UnitInit: present=9, absent=[], min=0.9930, max=1.0000, abs_spread=0.0070, rel_spread=0.0070, stable=True
  UnitOwnerChange: present=8, absent=[2016], min=0.2231, max=0.2885, abs_spread=0.0653, rel_spread=0.2265, stable=True
    caveat: sparse / absent in years [2016] but present in 8 of 9 years; present-year spread within family threshold
  UnitPositions: present=9, absent=[], min=0.9905, max=0.9996, ab

### V4.2 -- gameVersion cardinality + cohort strategy

If `gameVersion` cardinality > 20, group by major.minor prefix
(e.g., `5.0`, `4.10`, `4.11`); otherwise use raw gameVersion.
Grouping rule is recorded explicitly so it can be reproduced.

In [36]:
v4_2_gv_df = run_q(
    "v4_2_gameversion_cardinality",
    """
    SELECT metadata.gameVersion AS game_version,
           COUNT(*) AS n_replays
    FROM replays_meta_raw
    GROUP BY game_version
    ORDER BY n_replays DESC
    """,
)
N_DISTINCT_GAMEVERSION = len(v4_2_gv_df)
USE_GROUPED_COHORT = N_DISTINCT_GAMEVERSION > 20
COHORT_REGEX = r'^([0-9]+\.[0-9]+)'
if USE_GROUPED_COHORT:
    COHORT_LABEL = "major_minor"
    COHORT_RULE = (
        f"regexp_extract(metadata.gameVersion, '{COHORT_REGEX}', 1)"
    )
else:
    COHORT_LABEL = "raw_gameVersion"
    COHORT_RULE = "metadata.gameVersion"
print(
    f"Distinct gameVersion: {N_DISTINCT_GAMEVERSION}; "
    f"grouped={USE_GROUPED_COHORT}; cohort_label={COHORT_LABEL}; "
    f"cohort_rule='{COHORT_RULE}'"
)


def cohort_expr(prefix: str = "") -> str:
    """Build the cohort SQL expression with optional table-alias prefix."""
    base = f"{prefix}metadata.gameVersion"
    if USE_GROUPED_COHORT:
        return f"regexp_extract({base}, '{COHORT_REGEX}', 1)"
    return base

Distinct gameVersion: 46; grouped=True; cohort_label=major_minor; cohort_rule='regexp_extract(metadata.gameVersion, '^([0-9]+\.[0-9]+)', 1)'


### V4.2 -- per-cohort replay totals + non-trivial cohort selection

Non-trivial cohort = cohort holding > 5% of replays. Falsifier
threshold (`>=2 keys differ between two cohorts`) only applies to
non-trivial cohorts.

In [37]:
v4_2_cohort_totals_df = run_q(
    "v4_2_cohort_replay_totals",
    f"""
    SELECT {cohort_expr()} AS cohort,
           COUNT(*) AS n_replays
    FROM replays_meta_raw
    GROUP BY cohort
    ORDER BY n_replays DESC
    """,
)
total_replays = int(v4_2_cohort_totals_df["n_replays"].sum())
v4_2_cohort_totals_df["pct"] = (
    v4_2_cohort_totals_df["n_replays"] / total_replays
)
v4_2_cohort_totals_df["non_trivial"] = v4_2_cohort_totals_df["pct"] > 0.05
print(f"=== V4.2 cohort replay totals ({COHORT_LABEL}) ===")
print(v4_2_cohort_totals_df.to_string(index=False))
NON_TRIVIAL_COHORTS = sorted(
    v4_2_cohort_totals_df.loc[
        v4_2_cohort_totals_df["non_trivial"], "cohort"
    ].astype(str).tolist()
)
print(f"\nNon-trivial cohorts (>5% of {total_replays}): "
      f"{NON_TRIVIAL_COHORTS}")

=== V4.2 cohort replay totals (major_minor) ===
cohort  n_replays      pct  non_trivial
   5.0      10955 0.489281         True
   4.8       1762 0.078696         True
   4.9       1095 0.048906        False
  4.10       1029 0.045958        False
  4.12       1020 0.045556        False
  4.11        795 0.035507        False
   4.4        753 0.033631        False
   4.2        609 0.027200        False
   4.6        514 0.022957        False
   3.1        495 0.022108        False
   4.1        482 0.021527        False
   4.7        416 0.018580        False
  3.10        409 0.018267        False
   4.3        406 0.018133        False
  3.12        405 0.018088        False
  3.14        332 0.014828        False
  3.19        294 0.013131        False
  3.17        236 0.010540        False
  3.16        188 0.008397        False
   4.0        135 0.006029        False
   3.4         60 0.002680        False

Non-trivial cohorts (>5% of 22390): ['4.8', '5.0']


### V4.2 -- Pass A: 1% Bernoulli key-sets per (evtTypeName, cohort)

`TABLESAMPLE BERNOULLI(1) REPEATABLE(42)` for reproducibility.
Key-set extracted via `UNNEST(json_keys(event_data))`.

In [38]:
v4_2_pass_a_df = run_q(
    "v4_2_pass_a_keysets",
    f"""
    WITH samp AS (
        SELECT te.evtTypeName, te.event_data, te.filename
        FROM tracker_events_raw te
        TABLESAMPLE BERNOULLI(1%) REPEATABLE(42)
    ),
    joined AS (
        SELECT s.evtTypeName, s.event_data,
               {cohort_expr('rm.')} AS cohort
        FROM samp s JOIN replays_meta_raw rm USING (filename)
    ),
    ec AS (
        SELECT evtTypeName, cohort, COUNT(*) AS n_events
        FROM joined GROUP BY 1, 2
    ),
    un AS (
        SELECT evtTypeName, cohort, UNNEST(json_keys(event_data)) AS k
        FROM joined
    ),
    ks AS (
        SELECT evtTypeName, cohort, list(DISTINCT k ORDER BY k) AS keys
        FROM un GROUP BY evtTypeName, cohort
    )
    SELECT ks.evtTypeName, ks.cohort, ks.keys, ec.n_events
    FROM ks JOIN ec USING (evtTypeName, cohort)
    ORDER BY ks.evtTypeName, ks.cohort
    """,
)
print(f"Pass A rows: {len(v4_2_pass_a_df)}")
print(v4_2_pass_a_df[["evtTypeName", "cohort", "n_events"]]
      .to_string(index=False))

Pass A rows: 207
    evtTypeName cohort  n_events
    PlayerSetup    3.1        12
    PlayerSetup   3.10         6
    PlayerSetup   3.12        12
    PlayerSetup   3.14         9
    PlayerSetup   3.16         8
    PlayerSetup   3.17         6
    PlayerSetup   3.19         6
    PlayerSetup    4.0         3
    PlayerSetup    4.1        13
    PlayerSetup   4.10        15
    PlayerSetup   4.11        12
    PlayerSetup   4.12        25
    PlayerSetup    4.2        15
    PlayerSetup    4.3         4
    PlayerSetup    4.4        17
    PlayerSetup    4.6        13
    PlayerSetup    4.7         8
    PlayerSetup    4.8        34
    PlayerSetup    4.9        21
    PlayerSetup    5.0       225
    PlayerStats    3.1       912
    PlayerStats   3.10       788
    PlayerStats   3.12       780
    PlayerStats   3.14       658
    PlayerStats   3.16       399
    PlayerStats   3.17       422
    PlayerStats   3.19       643
    PlayerStats    3.4       131
    PlayerStats    4.0    

### V4.2 -- identify Pass A under-sampled cells

A `(evtTypeName, cohort)` cell is under-sampled if Pass A has
< 1000 events AND the cohort is non-trivial. These cells need
Pass B confirmation before any "unstable" verdict.

In [39]:
under = v4_2_pass_a_df[
    (v4_2_pass_a_df["n_events"] < 1000) &
    (v4_2_pass_a_df["cohort"].astype(str).isin(NON_TRIVIAL_COHORTS))
]
print(f"Under-sampled (Pass A) cells: {len(under)}")
if len(under):
    print(under[["evtTypeName", "cohort", "n_events"]]
          .to_string(index=False))
under_event_types = sorted(under["evtTypeName"].unique().tolist())
under_cohorts = sorted(under["cohort"].astype(str).unique().tolist())
print(f"\nUnder-sampled event types: {under_event_types}")
print(f"Under-sampled cohorts:    {under_cohorts}")

Under-sampled (Pass A) cells: 6
    evtTypeName cohort  n_events
    PlayerSetup    4.8        34
    PlayerSetup    5.0       225
UnitOwnerChange    4.8        51
UnitOwnerChange    5.0       353
  UnitPositions    4.8       670
        Upgrade    4.8       547

Under-sampled event types: ['PlayerSetup', 'UnitOwnerChange', 'UnitPositions', 'Upgrade']
Under-sampled cohorts:    ['4.8', '5.0']


### V4.2 -- Pass B: stratified resample for under-sampled cells

Per Amendment 6 + plan T06.3: for every under-sampled
`(evtTypeName, cohort)` cell, draw up to 10K rows by
`ROW_NUMBER() OVER (PARTITION BY evtTypeName, cohort ORDER BY loop)`.
This gives a sample large enough that key-set findings cannot be
dismissed as under-coverage.

In [40]:
if under_event_types and under_cohorts:
    et_list = ", ".join(f"'{e}'" for e in under_event_types)
    coh_list = ", ".join(f"'{c}'" for c in under_cohorts)
    v4_2_pass_b_df = run_q(
        "v4_2_pass_b_keysets",
        f"""
        WITH ranked AS (
            SELECT te.evtTypeName, te.event_data, te.loop,
                   {cohort_expr('rm.')} AS cohort,
                   ROW_NUMBER() OVER (
                       PARTITION BY te.evtTypeName, {cohort_expr('rm.')}
                       ORDER BY te.loop
                   ) AS rn
            FROM tracker_events_raw te
            JOIN replays_meta_raw rm USING (filename)
            WHERE te.evtTypeName IN ({et_list})
        ),
        sampled AS (
            SELECT evtTypeName, event_data, cohort
            FROM ranked
            WHERE rn <= 10000 AND cohort IN ({coh_list})
        ),
        ec AS (
            SELECT evtTypeName, cohort, COUNT(*) AS n_events
            FROM sampled GROUP BY 1, 2
        ),
        un AS (
            SELECT evtTypeName, cohort,
                   UNNEST(json_keys(event_data)) AS k
            FROM sampled
        ),
        ks AS (
            SELECT evtTypeName, cohort,
                   list(DISTINCT k ORDER BY k) AS keys
            FROM un GROUP BY evtTypeName, cohort
        )
        SELECT ks.evtTypeName, ks.cohort, ks.keys, ec.n_events
        FROM ks JOIN ec USING (evtTypeName, cohort)
        ORDER BY ks.evtTypeName, ks.cohort
        """,
    )
    print(f"Pass B rows: {len(v4_2_pass_b_df)}")
    print(v4_2_pass_b_df[["evtTypeName", "cohort", "n_events"]]
          .to_string(index=False))
else:
    v4_2_pass_b_df = pd.DataFrame(
        columns=["evtTypeName", "cohort", "keys", "n_events"]
    )
    sql_queries["v4_2_pass_b_keysets"] = (
        "-- skipped: no under-sampled cells in Pass A"
    )
    print("No under-sampled cells; Pass B skipped.")

Pass B rows: 8
    evtTypeName cohort  n_events
    PlayerSetup    4.8      3524
    PlayerSetup    5.0     10000
UnitOwnerChange    4.8      4610
UnitOwnerChange    5.0     10000
  UnitPositions    4.8     10000
  UnitPositions    5.0     10000
        Upgrade    4.8     10000
        Upgrade    5.0     10000


### V4.2 -- combine Pass A + Pass B into authoritative key-sets

For each `(evtTypeName, cohort)`, pick the larger sample. Record
`sample_strategy` per cell (pass_a_only / pass_b_stratified /
too_sparse).

In [41]:
authoritative: dict = {}
pass_a_lookup = {
    (r["evtTypeName"], str(r["cohort"])): {
        "keys": list(r["keys"]), "n_events": int(r["n_events"]),
    }
    for _, r in v4_2_pass_a_df.iterrows()
}
pass_b_lookup = {
    (r["evtTypeName"], str(r["cohort"])): {
        "keys": list(r["keys"]), "n_events": int(r["n_events"]),
    }
    for _, r in v4_2_pass_b_df.iterrows()
}
all_cells = set(pass_a_lookup) | set(pass_b_lookup)
for cell in all_cells:
    a = pass_a_lookup.get(cell)
    b = pass_b_lookup.get(cell)
    if b is not None and (a is None or b["n_events"] > a["n_events"]):
        authoritative[cell] = {**b, "sample_strategy": "pass_b_stratified"}
    elif a is not None and a["n_events"] >= 1000:
        authoritative[cell] = {**a, "sample_strategy": "pass_a_only"}
    elif a is not None:
        authoritative[cell] = {**a, "sample_strategy": "too_sparse"}
    else:
        authoritative[cell] = {**b, "sample_strategy": "pass_b_stratified"}
print(f"Authoritative cells: {len(authoritative)}")

Authoritative cells: 207


### V4.2 -- per-event-type stability check

`key_set_stable=False` iff any pair of *non-trivial* cohorts (each
>5% of the corpus, each with >=1000 events in the chosen sample)
differs by >=2 keys. `too_sparse_to_decide` if no cohort meets the
1000-event threshold under any sample strategy.

In [42]:
def stability_for(et: str) -> dict:
    """Compute key-set stability for one event type across cohorts."""
    cells = {c: v for c, v in authoritative.items() if c[0] == et}
    nt = {
        c[1]: v for c, v in cells.items()
        if c[1] in NON_TRIVIAL_COHORTS and v["n_events"] >= 1000
    }
    if len(nt) == 0:
        return {
            "key_set_stable": "too_sparse_to_decide",
            "non_trivial_cohorts_in_sample": [],
            "unstable_versions": [],
            "key_union_size": None,
            "max_pairwise_diff": None,
            "sample_strategy": "too_sparse",
        }
    cohorts = sorted(nt)
    union: set = set()
    for v in nt.values():
        union |= set(v["keys"])
    max_diff = 0
    unstable_pairs: list = []
    for i in range(len(cohorts)):
        for j in range(i + 1, len(cohorts)):
            c1, c2 = cohorts[i], cohorts[j]
            sym = set(nt[c1]["keys"]) ^ set(nt[c2]["keys"])
            if len(sym) > max_diff:
                max_diff = len(sym)
            if len(sym) >= 2:
                unstable_pairs.append({
                    "a": c1, "b": c2, "sym_diff_size": len(sym),
                    "sym_diff": sorted(sym),
                })
    strat = {v["sample_strategy"] for v in nt.values()}
    if strat == {"pass_a_only"}:
        ss = "pass_a_only"
    elif "pass_b_stratified" in strat:
        ss = "pass_b_stratified"
    else:
        ss = "mixed"
    return {
        "key_set_stable": (max_diff < 2),
        "non_trivial_cohorts_in_sample": cohorts,
        "unstable_versions": unstable_pairs,
        "key_union_size": len(union),
        "max_pairwise_diff": max_diff,
        "sample_strategy": ss,
    }

In [43]:
v4_per_event: dict = {}
for et in event_types_observed:
    cov = year_coverage_summary[et]
    stab = stability_for(et)
    v4_per_event[et] = {
        "coverage_stable": cov["coverage_stable_by_year"],
        "min_year_coverage": cov["min_year_coverage"],
        "max_year_coverage": cov["max_year_coverage"],
        "coverage_spread": cov["coverage_spread"],
        "coverage_relative_spread": cov["coverage_relative_spread"],
        "years_present": cov["years_present"],
        "years_absent": cov["years_absent"],
        "stability_basis": cov["stability_basis"],
        "coverage_caveat": cov["coverage_caveat"],
        "key_set_stable": stab["key_set_stable"],
        "non_trivial_cohorts_in_sample":
            stab["non_trivial_cohorts_in_sample"],
        "unstable_versions": stab["unstable_versions"],
        "key_union_size": stab["key_union_size"],
        "max_pairwise_diff": stab["max_pairwise_diff"],
        "sample_strategy": stab["sample_strategy"],
        "feature_eligibility_caveat": cov["feature_eligibility_caveat"],
        "caveat": None,
    }
    if stab["key_set_stable"] == "too_sparse_to_decide":
        v4_per_event[et]["caveat"] = (
            "rare-family safeguard (Amendment 6): no cohort reached "
            "1000-event threshold under Pass A or Pass B"
        )
print("=== V4 per-event verdict ===")
for et in event_types_observed:
    v = v4_per_event[et]
    print(
        f"  {et}: cov_stable={v['coverage_stable']}, "
        f"key_stable={v['key_set_stable']}, "
        f"sample={v['sample_strategy']}, "
        f"max_diff={v['max_pairwise_diff']}"
    )

=== V4 per-event verdict ===
  PlayerSetup: cov_stable=True, key_stable=True, sample=pass_b_stratified, max_diff=0
  PlayerStats: cov_stable=True, key_stable=True, sample=pass_a_only, max_diff=0
  UnitBorn: cov_stable=True, key_stable=True, sample=pass_a_only, max_diff=0
  UnitDied: cov_stable=True, key_stable=True, sample=pass_a_only, max_diff=0
  UnitDone: cov_stable=True, key_stable=True, sample=pass_a_only, max_diff=0
  UnitInit: cov_stable=True, key_stable=True, sample=pass_a_only, max_diff=0
  UnitOwnerChange: cov_stable=True, key_stable=True, sample=pass_b_stratified, max_diff=0
  UnitPositions: cov_stable=True, key_stable=True, sample=pass_b_stratified, max_diff=0
  UnitTypeChange: cov_stable=True, key_stable=True, sample=pass_a_only, max_diff=0
  Upgrade: cov_stable=True, key_stable=True, sample=pass_b_stratified, max_diff=0


### V4 verdict assembly + result

Result discipline (per T06 step 6): one sparse event family does NOT
fail V4 unless it blocks ALL planned tracker-derived feature families.
A `too_sparse_to_decide` rare family is recorded but does not invert
`result` to FAIL.

In [44]:
stable_events = [
    et for et in event_types_observed
    if v4_per_event[et]["coverage_stable"] is True
    and v4_per_event[et]["key_set_stable"] is True
]
unstable_events = [
    et for et in event_types_observed
    if v4_per_event[et]["coverage_stable"] is False
    or v4_per_event[et]["key_set_stable"] is False
]
sparse_events = [
    et for et in event_types_observed
    if v4_per_event[et]["coverage_stable"] == "too_sparse_to_decide"
    or v4_per_event[et]["key_set_stable"] == "too_sparse_to_decide"
]
caveated_events = [
    et for et in event_types_observed
    if v4_per_event[et]["feature_eligibility_caveat"] is not None
    or v4_per_event[et]["coverage_caveat"] is not None
]
all_have_verdicts = all(
    et in v4_per_event for et in event_types_observed
)

if unstable_events:
    v4_result = "PASS_WITH_CAVEAT"
    v4_caveat = (
        f"unstable event families: {unstable_events}; "
        f"stable: {len(stable_events)}; too_sparse: {sparse_events}; "
        f"caveated: {caveated_events}"
    )
elif sparse_events:
    v4_result = "PASS_WITH_CAVEAT"
    v4_caveat = (
        f"all checked families stable; rare-family safeguard kept "
        f"{sparse_events} as too_sparse_to_decide (not failed); "
        f"caveated: {caveated_events}"
    )
elif caveated_events:
    # Sparse / occasionally-absent families pass the spread threshold
    # within their basis, but carry a feature_eligibility caveat for V8.
    # Per T06 hygiene-pass step 3: PASS only if sparse-family caveats
    # provably do not affect planned Phase 02 families. T06 cannot
    # validate Phase 02 plans, so default to PASS_WITH_CAVEAT.
    v4_result = "PASS_WITH_CAVEAT"
    v4_caveat = (
        f"all {len(event_types_observed)} event families stable; "
        f"sparse / occasionally-absent caveats on: {caveated_events}; "
        f"V8 must propagate these as feature_eligibility caveats"
    )
else:
    v4_result = "PASS"
    v4_caveat = (
        f"all {len(event_types_observed)} event families stable "
        f"across years and non-trivial cohorts; no sparse-family caveats"
    )

verdicts["V4"] = {
    "hypothesis": (
        "all 10 tracker event families have stable presence and JSON "
        "key structure across the 2016-2024 SC2EGSet corpus"
    ),
    "falsifier": (
        "any planned-feature family missing/unstable across major "
        "year/gameVersion cohorts; key-set differs >=2 keys between "
        "non-trivial cohorts (>5% each); rare-event under-sampling "
        "blocks decision"
    ),
    "result": v4_result,
    "rare_family_safeguard_applied": True,
    "protocol88500_snapshot_caveat": (
        "T02 used protocol88500.py as a recent reference snapshot only; "
        "V4 stability is empirical across observed corpus, NOT a claim "
        "about all SC2 protocols ever shipped"
    ),
    "cohort_strategy": {
        "n_distinct_gameversion": N_DISTINCT_GAMEVERSION,
        "use_grouped_cohort": USE_GROUPED_COHORT,
        "cohort_label": COHORT_LABEL,
        "cohort_rule": COHORT_RULE,
        "non_trivial_cohorts": NON_TRIVIAL_COHORTS,
        "non_trivial_threshold_pct": 0.05,
    },
    "year_coverage_summary": year_coverage_summary,
    "per_event": v4_per_event,
    "stable_event_types": stable_events,
    "unstable_event_types": unstable_events,
    "too_sparse_event_types": sparse_events,
    "caveated_event_types": caveated_events,
    "v8_carry_forward_caveats": {
        et: v4_per_event[et]["feature_eligibility_caveat"]
        for et in event_types_observed
        if v4_per_event[et]["feature_eligibility_caveat"] is not None
    },
    "all_event_types_have_verdict": all_have_verdicts,
    "notes": v4_caveat,
}
print("=== V4 verdict ===")
print(f"  result: {verdicts['V4']['result']}")
print(f"  stable: {len(stable_events)} -> {stable_events}")
print(f"  unstable: {len(unstable_events)} -> {unstable_events}")
print(f"  too_sparse: {len(sparse_events)} -> {sparse_events}")
print(f"  caveated (V8 carry-forward): {len(caveated_events)} -> "
      f"{caveated_events}")
print(f"  notes: {verdicts['V4']['notes']}")

=== V4 verdict ===
  result: PASS_WITH_CAVEAT
  stable: 10 -> ['PlayerSetup', 'PlayerStats', 'UnitBorn', 'UnitDied', 'UnitDone', 'UnitInit', 'UnitOwnerChange', 'UnitPositions', 'UnitTypeChange', 'Upgrade']
  unstable: 0 -> []
  too_sparse: 0 -> []
  caveated (V8 carry-forward): 1 -> ['UnitOwnerChange']
  notes: all 10 event families stable; sparse / occasionally-absent caveats on: ['UnitOwnerChange']; V8 must propagate these as feature_eligibility caveats


### V4 consistency assertions

In [45]:
assert all_have_verdicts, (
    f"missing V4 verdict for some event types: "
    f"{set(event_types_observed) - set(v4_per_event)}"
)
assert len(v4_per_event) == 10, (
    f"expected 10 event types, got {len(v4_per_event)}"
)
for et, v in v4_per_event.items():
    assert v["coverage_stable"] in (True, False, "too_sparse_to_decide"), (
        f"{et}: bad coverage_stable={v['coverage_stable']}"
    )
    assert v["key_set_stable"] in (True, False, "too_sparse_to_decide"), (
        f"{et}: bad key_set_stable={v['key_set_stable']}"
    )
    assert v["sample_strategy"] in (
        "pass_a_only", "pass_b_stratified", "mixed", "too_sparse"
    ), f"{et}: bad sample_strategy={v['sample_strategy']}"
print("=== V4 consistency: OK ===")
print(f"  10 event types verdict: stable={len(stable_events)}, "
      f"unstable={len(unstable_events)}, "
      f"too_sparse={len(sparse_events)}")

=== V4 consistency: OK ===
  10 event types verdict: stable=10, unstable=0, too_sparse=0


## V4 result summary

- **Per-year coverage:** 9 years (2016-2024). The
  `v4_1_per_year_coverage` query uses `years CROSS JOIN event_types
  LEFT JOIN counts` so every `(year, event_type)` cell is materialised
  even when zero -- absence is visible to the classifier, not silently
  dropped. Common event families (UnitBorn, UnitDied, UnitTypeChange,
  PlayerStats, UnitInit, UnitDone, UnitPositions, Upgrade, PlayerSetup)
  appear at near-100% replay coverage every year (absolute spread
  <= 5%). **UnitOwnerChange is schema-stable but coverage-sparse:**
  it is absent in the 2016 slice and appears in roughly one quarter
  of replays in later years. This supports descriptive stability of
  the event schema where present, but not robust use as a broadly
  available feature family without caveats. The classifier records
  `coverage_caveat` and `feature_eligibility_caveat:
  sparse_event_family_not_broadly_available` as machine-readable V8
  carry-forward fields under `verdicts["V4"]["v8_carry_forward_caveats"]`.
- **Cohort strategy:** 46 distinct `gameVersion` strings -> grouped
  by `major.minor` prefix (e.g., `5.0`, `4.10`, `4.11`). Non-trivial
  cohorts are those holding > 5% of replays.
- **Pass A / Pass B:** 1% Bernoulli sample with `REPEATABLE(42)` for
  reproducibility. Cells under-sampled in Pass A (< 1000 events in a
  non-trivial cohort) trigger Pass B stratified resample
  (`ROW_NUMBER() OVER (PARTITION BY evtTypeName, cohort ORDER BY loop)
  <= 10000`). Cells still < 1000 after Pass B are
  `coverage_too_sparse_for_stability_decision`, NOT `unstable`.
- **Key-set stability:** assessed only for non-trivial cohorts each
  holding >= 1000 events in the chosen sample. Pairwise symmetric
  difference >= 2 keys = `key_set_stable=False`.
- **Carry-forward (T02):** the s2protocol `protocol88500.py` snapshot
  is a *recent* reference, NOT a proof of historical stability across
  the 2016-2024 corpus. V4 stability is empirical, sample-confirmed
  for the corpus actually observed.
- **V4 verdict:** see printed result above.

## Out of scope for T06 (this notebook execution)

- V5..V8 are deferred to T07..T10 per `planning/current_plan.md`.
- Final `.md` / `.json` / `.csv` artifacts under
  `reports/artifacts/01_exploration/03_profiling/` are produced
  atomically in T11, NOT in T06.
- STEP_STATUS / PIPELINE_SECTION_STATUS / PHASE_STATUS updates are
  T11-atomic per WARNING-3 + WARNING-4 fold.
- research_log entry is T11.
- thesis chapter prose, AoE2 datasets, specs, pyproject, poetry,
  Phase 02 feature engineering -- all out of scope.